In [ ]:
# @title Choose data and portfolio assumptions
from pathlib import Path
import os

DATA_MODE = "yahoo"       # "demo", "yahoo", or "csv"
CSV_PATH = None          # For example "/content/adjusted_prices.csv"
USE_FRED = False         # Requires a FRED_API_KEY secret; otherwise use the stated fixed rate.
REFRESH_CACHE = False    # True explicitly replaces cached provider snapshots.
DOWNLOAD_RESULTS = False  # Set True to trigger Colab's file download in the final cell.

START = "2015-01-01"
END = "2026-01-01"       # Exclusive
INITIAL_CAPITAL = 100_000.0
COST_BPS = 5.0           # Per dollar bought OR sold
REBALANCE = "monthly"    # monthly, quarterly, annual, none
FIXED_RF_ANNUAL = 0.02   # Effective annual assumption, not historical cash returns
WEIGHTS = {"SPY": .30, "VEA": .15, "VWO": .05, "AGG": .20,
           "IEF": .10, "TIP": .05, "GLD": .10, "VNQ": .05}
BENCHMARK_WEIGHTS = {"SPY": .60, "AGG": .40}
PROJECT_ROOT = Path(os.environ.get("PORTFOLIO_PROJECT_ROOT", "portfolio_lab_project")).resolve()
print("Data mode:", DATA_MODE)
print("Project folder:", PROJECT_ROOT)


In [ ]:
# @title Install only the dependencies this run needs
import importlib.util
import importlib.metadata
import re
import subprocess
import sys

requirements = {
    "numpy": ((1, 26), 3, "numpy>=1.26,<3"),
    "pandas": ((2, 2), 3, "pandas>=2.2,<3"),
    "matplotlib": ((3, 8), 4, "matplotlib>=3.8,<4"),
}
to_install = []
for package, (minimum, maximum_major, requirement) in requirements.items():
    if importlib.util.find_spec(package) is None:
        to_install.append(requirement)
    else:
        match = re.match(r"(\d+)\.(\d+)", importlib.metadata.version(package))
        installed = tuple(map(int, match.groups()))
        if installed < minimum or installed[0] >= maximum_major:
            to_install.append(requirement)
if DATA_MODE == "yahoo" and importlib.util.find_spec("yfinance") is None:
    to_install.append("yfinance>=0.2.54,<2")
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
print("Dependencies ready:", list(requirements), "+ yfinance" if DATA_MODE == "yahoo" else "")


In [ ]:
# @title Create the reusable project structure
import sys
import logging

for folder in ("portfolio_lab", "tests", "data/cache", "reports", "docs", "examples",
               ".github/workflows"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / "portfolio_lab/__init__.py").write_text(
    '"""Auditable portfolio research."""\n__version__ = "1.0.0"\n', encoding="utf-8")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
print("Working directory:", Path.cwd())


In [ ]:
%%writefile portfolio_lab/config.py
"""Validated configuration. Weights are fractions; transaction costs are basis points."""

from dataclasses import asdict, dataclass, field
import math
import pandas as pd

DEFAULT_WEIGHTS = {
    "SPY": 0.30, "VEA": 0.15, "VWO": 0.05, "AGG": 0.20,
    "IEF": 0.10, "TIP": 0.05, "GLD": 0.10, "VNQ": 0.05,
}
ASSET_NAMES = {
    "SPY": "US equities", "VEA": "Developed ex-US equities",
    "VWO": "Emerging-market equities", "AGG": "US aggregate bonds",
    "IEF": "Intermediate Treasuries", "TIP": "Inflation-linked bonds",
    "GLD": "Gold", "VNQ": "US listed real estate",
}


@dataclass(frozen=True)
class Config:
    """All portfolio assumptions in one place; no hidden normalization of weights."""

    start: str = "2015-01-01"
    end: str = "2026-01-01"  # Exclusive, matching Yahoo's historical download API.
    weights: dict[str, float] = field(default_factory=lambda: DEFAULT_WEIGHTS.copy())
    benchmark_weights: dict[str, float] = field(
        default_factory=lambda: {"SPY": 0.60, "AGG": 0.40}
    )
    initial_capital: float = 100_000.0
    rebalance: str = "monthly"
    cost_bps: float = 5.0  # Applied to EACH dollar bought or sold.
    annualization: int = 252
    rolling_window: int = 126
    min_observations: int = 60
    fixed_rf_annual: float = 0.02  # Explicit assumption, not a historical observation.
    confidence: float = 0.95
    seed: int = 42

    def __post_init__(self):
        start, end = pd.Timestamp(self.start), pd.Timestamp(self.end)
        if pd.isna(start) or pd.isna(end) or start >= end:
            raise ValueError("start must be before exclusive end.")
        if start.tz is not None or end.tz is not None:
            raise ValueError("Use timezone-naive date strings in Config.")
        for name, weights in (("weights", self.weights),
                              ("benchmark_weights", self.benchmark_weights)):
            if not weights or any(not isinstance(k, str) or not k for k in weights):
                raise ValueError(f"{name} must contain nonempty ticker names.")
            if any(not math.isfinite(v) or v < 0 for v in weights.values()):
                raise ValueError(f"{name} must be finite and nonnegative.")
            if not math.isclose(sum(weights.values()), 1.0, abs_tol=1e-10):
                raise ValueError(f"{name} must sum to 1.0; received {sum(weights.values())}.")
        if not math.isfinite(self.initial_capital) or self.initial_capital <= 0:
            raise ValueError("initial_capital must be positive and finite.")
        if not math.isfinite(self.cost_bps) or not 0 <= self.cost_bps <= 1000:
            raise ValueError("cost_bps must be between 0 and 1,000.")
        if self.rebalance not in {"monthly", "quarterly", "annual", "none"}:
            raise ValueError("Unsupported rebalance frequency.")
        if self.annualization <= 1 or self.rolling_window < 2 or self.min_observations < 3:
            raise ValueError("Invalid annualization, window, or observation count.")
        if not math.isfinite(self.fixed_rf_annual) or self.fixed_rf_annual <= -1:
            raise ValueError("The effective annual risk-free assumption must exceed -100%.")
        if not 0.5 < self.confidence < 1:
            raise ValueError("confidence must lie between 0.5 and 1.")

    @property
    def tickers(self) -> list[str]:
        return sorted(set(self.weights) | set(self.benchmark_weights))

    def to_dict(self) -> dict:
        return asdict(self)


In [ ]:
%%writefile portfolio_lab/data.py
"""Price adapters, explicit provenance, fail-closed cleaning, and risk-free proxies."""

from dataclasses import dataclass
from datetime import datetime, timezone
from hashlib import sha256
from io import StringIO
from pathlib import Path
import json
import logging
import time
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

from .config import Config

LOG = logging.getLogger(__name__)


class DataError(RuntimeError):
    """Actionable provider, schema, or data-quality failure."""


@dataclass
class DataBundle:
    prices: pd.DataFrame
    metadata: dict


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def atomic_write(path: Path, content: str) -> None:
    """Replace a completed local file atomically; never leave a partial cache."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def _cache_paths(directory: Path, query: dict) -> tuple[Path, Path]:
    key = sha256(json.dumps(query, sort_keys=True).encode()).hexdigest()[:20]
    return directory / f"{key}.csv", directory / f"{key}.json"


def _load_cache(directory: Path, query: dict) -> DataBundle | None:
    data_path, meta_path = _cache_paths(directory, query)
    if not data_path.exists() or not meta_path.exists():
        return None
    try:
        raw = data_path.read_bytes()
        metadata = json.loads(meta_path.read_text())
        if sha256(raw).hexdigest() != metadata["sha256"]:
            raise ValueError("checksum mismatch")
        prices = pd.read_csv(StringIO(raw.decode()), index_col=0, parse_dates=True)
    except (ValueError, KeyError, OSError) as exc:
        raise DataError("Cache is corrupt. Delete its CSV/JSON pair or force refresh.") from exc
    metadata["cache_hit"] = True
    return DataBundle(prices, metadata)


def _save_cache(directory: Path, query: dict, bundle: DataBundle) -> None:
    data_path, meta_path = _cache_paths(directory, query)
    csv = bundle.prices.to_csv(index_label="Date", float_format="%.12g")
    bundle.metadata["sha256"] = sha256(csv.encode()).hexdigest()
    atomic_write(data_path, csv)
    atomic_write(meta_path, json.dumps(bundle.metadata, indent=2))


def extract_adjusted_close(raw: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    """Accept Yahoo's flat or either orientation of two-level column schemas."""
    if raw is None or raw.empty:
        raise DataError("Yahoo returned no rows. Check dates, symbols, and connectivity.")
    if isinstance(raw.columns, pd.MultiIndex):
        selected = None
        for level in range(raw.columns.nlevels):
            if "Adj Close" in raw.columns.get_level_values(level):
                selected = raw.xs("Adj Close", axis=1, level=level)
                break
        if selected is None:
            raise DataError("Adjusted close is absent; raw close is not a safe substitute.")
        prices = selected.copy()
    elif "Adj Close" in raw.columns and len(tickers) == 1:
        prices = raw[["Adj Close"]].rename(columns={"Adj Close": tickers[0]})
    else:
        raise DataError("Unrecognized Yahoo schema or missing adjusted prices.")
    missing = set(tickers) - set(prices.columns)
    if missing:
        raise DataError(f"No adjusted prices for: {sorted(missing)}.")
    return prices.loc[:, tickers]


def fetch_yahoo(config: Config, cache_dir: Path, refresh: bool = False,
                downloader=None, max_attempts: int = 3) -> DataBundle:
    """Fetch adjusted close explicitly; do not add dividends to these returns again.

    An injected downloader supports deterministic provider-contract tests. The cache
    is intentionally a frozen snapshot until refresh=True; metadata exposes its age.
    """
    query = {"source": "yahoo", "field": "Adj Close", "tickers": config.tickers,
             "start": config.start, "end_exclusive": config.end}
    if not refresh:
        cached = _load_cache(cache_dir, query)
        if cached is not None:
            return cached
    if downloader is None:
        try:
            import yfinance as yf
        except ImportError:
            raise DataError("Yahoo mode requires yfinance. Install requirements-live.txt.") from None
        downloader = yf.download
    if max_attempts < 1:
        raise ValueError("max_attempts must be positive.")
    last_type = "unknown"
    for attempt in range(max_attempts):
        try:
            raw = downloader(config.tickers, start=config.start, end=config.end,
                             auto_adjust=False, actions=True, progress=False,
                             threads=False, group_by="column", timeout=20)
            prices = extract_adjusted_close(raw, config.tickers)
            if prices.notna().sum().min() == 0:
                raise DataError("One or more ticker downloads contain no observations.")
            bundle = DataBundle(prices, {
                **query, "synthetic": False, "retrieved_at_utc": utc_now(),
                "cache_hit": False, "adjustment": "Yahoo adjusted close; no added dividends",
            })
            _save_cache(cache_dir, query, bundle)
            return bundle
        except Exception as exc:
            # A retry is bounded and never substitutes fabricated prices.
            last_type = type(exc).__name__
            LOG.warning("Yahoo attempt %s/%s failed (%s).", attempt + 1, max_attempts, last_type)
            if attempt + 1 < max_attempts:
                time.sleep(min(2 ** attempt, 4))
    raise DataError(
        f"Yahoo failed after {max_attempts} attempts ({last_type}). "
        "Check provider access, tickers, or use an explicit cached CSV/demo mode."
    )


def generate_demo(config: Config) -> DataBundle:
    """Deterministic synthetic total-return indices. These are NOT ETF histories."""
    dates = pd.bdate_range(config.start, pd.Timestamp(config.end) - pd.Timedelta(days=1))
    if len(dates) < config.min_observations:
        raise DataError("Demo date range is too short.")
    rng = np.random.default_rng(config.seed)
    # Common equity, duration, and gold factors; residuals preserve diversification.
    mapping = {
        "SPY": [0.15, 0.00, 0.00], "VEA": [0.14, 0.00, 0.00],
        "VWO": [0.17, 0.00, 0.00], "AGG": [0.01, 0.045, 0.00],
        "IEF": [-0.01, 0.065, 0.00], "TIP": [0.02, 0.04, 0.01],
        "GLD": [0.01, 0.01, 0.13], "VNQ": [0.13, 0.03, 0.00],
    }
    count = len(dates) - 1
    factors = rng.standard_t(7, size=(count, 3)) / np.sqrt(7 / 5)
    factors /= np.sqrt(252)
    factors[int(count * .43):int(count * .46), 0] -= .008
    factors[int(count * .72):int(count * .77), :2] -= .004
    values = {}
    for ticker in config.tickers:
        exposure = np.array(mapping.get(ticker, [.10, .03, .02]))
        drift = .055 if exposure[0] > .1 else .025
        log_returns = drift / 252 + factors @ exposure
        log_returns += rng.normal(0, .04 / np.sqrt(252), size=count)
        values[ticker] = 100 * np.exp(np.r_[0.0, np.cumsum(log_returns)])
    return DataBundle(pd.DataFrame(values, index=dates), {
        "source": "synthetic demo", "synthetic": True, "seed": config.seed,
        "generated_at_utc": utc_now(),
        "description": "Generated total-return indices; ticker labels are illustrative.",
        "calendar": "Weekdays only; exchange holidays are not modeled.",
    })


def load_csv(path: Path) -> DataBundle:
    """CSV contract: Date column followed by USD adjusted total-return price columns."""
    if not path.is_file():
        raise DataError(f"CSV does not exist: {path}")
    try:
        prices = pd.read_csv(path, index_col="Date", parse_dates=True)
    except (ValueError, OSError) as exc:
        raise DataError("CSV needs a Date column and one numeric column per ticker.") from exc
    return DataBundle(prices, {"source": "user CSV", "synthetic": False,
                              "file_name": path.name,
                              "sha256": sha256(path.read_bytes()).hexdigest(),
                              "adjustment": "User-declared adjusted total-return prices"})


def clean_prices(raw: pd.DataFrame, config: Config) -> tuple[pd.DataFrame, dict]:
    """Trim to common history; never forward-fill prices or turn missing data into 0% returns."""
    if raw.empty or raw.columns.duplicated().any():
        raise DataError("Prices are empty or contain duplicate columns.")
    missing = set(config.tickers) - set(raw.columns)
    if missing:
        raise DataError(f"Missing required symbols: {sorted(missing)}.")
    prices = raw.loc[:, config.tickers].copy()
    try:
        index = pd.DatetimeIndex(pd.to_datetime(prices.index, errors="raise"))
    except (ValueError, TypeError):
        raise DataError("Invalid price timestamps.") from None
    if index.hasnans:
        raise DataError("Price timestamps contain NaT.")
    if index.tz is not None:
        index = index.tz_localize(None)  # Preserve exchange-local daily labels.
    prices.index = index.normalize()
    if prices.index.duplicated().any():
        raise DataError("Duplicate dates are ambiguous; resolve them in the source.")
    prices = prices.sort_index()
    prices = prices.loc[(prices.index >= pd.Timestamp(config.start)) &
                        (prices.index < pd.Timestamp(config.end))]
    if prices.empty:
        raise DataError("No prices in the configured interval.")
    try:
        prices = prices.apply(pd.to_numeric, errors="raise")
    except (ValueError, TypeError):
        raise DataError("Prices contain nonnumeric values.") from None
    array = prices.to_numpy(dtype=float)
    if np.isinf(array).any() or ((array <= 0) & ~np.isnan(array)).any():
        raise DataError("Observed prices must be finite and strictly positive.")
    if prices.notna().sum().min() == 0:
        raise DataError("A required asset has no valid history.")
    first = max(prices[c].first_valid_index() for c in prices)
    last = min(prices[c].last_valid_index() for c in prices)
    original_count = len(prices)
    prices = prices.loc[first:last]
    if len(prices) < config.min_observations:
        raise DataError(f"Need at least {config.min_observations} common price observations.")
    if prices.isna().any().any():
        cells = prices.isna().stack()
        example = [str(x) for x in cells[cells].index[:5]]
        raise DataError(f"Interior price gaps are not imputed. Example cells: {example}")
    largest_gap = int(prices.index.to_series().diff().dt.days.max())
    if largest_gap > 7:
        raise DataError(f"A {largest_gap}-day calendar gap needs investigation.")
    returns = prices.pct_change(fill_method=None).iloc[1:]
    extremes = returns.abs().gt(.35)
    warnings = []
    if original_count != len(prices):
        warnings.append(f"Trimmed {original_count - len(prices)} rows to common asset history.")
    if extremes.any().any():
        warnings.append(f"{int(extremes.sum().sum())} daily moves exceed 35%; inspect adjustment quality.")
    # Record, rather than hide, stale tails relative to the requested end.
    tail_days = (pd.Timestamp(config.end) - prices.index[-1]).days
    if tail_days > 7:
        warnings.append(f"Data ends {tail_days} days before the requested exclusive end.")
    audit = {
        "rows": len(prices), "return_observations": len(prices) - 1,
        "assets": len(prices.columns), "first_date": str(prices.index[0].date()),
        "last_date": str(prices.index[-1].date()), "trimmed_rows": original_count - len(prices),
        "interior_missing_cells": 0, "largest_calendar_gap_days": largest_gap,
        "warnings": warnings,
        "calendar_policy": "Observed common sessions; no exchange-calendar reconstruction.",
    }
    return prices.astype(float), audit


def fetch_fred_yields(config: Config, api_key: str, cache_dir: Path,
                      refresh: bool = False) -> DataBundle:
    """DGS3MO annual yields in percent; API credentials are never stored or logged."""
    if not api_key:
        raise DataError("FRED mode requires FRED_API_KEY in your environment/Colab secrets.")
    start = (pd.Timestamp(config.start) - pd.Timedelta(days=30)).strftime("%Y-%m-%d")
    query = {"source": "FRED", "series": "DGS3MO", "start": start, "end": config.end}
    if not refresh:
        cached = _load_cache(cache_dir, query)
        if cached is not None:
            return cached
    params = {"series_id": "DGS3MO", "api_key": api_key, "file_type": "json",
              "observation_start": start, "observation_end": config.end}
    url = "https://api.stlouisfed.org/fred/series/observations?" + urlencode(params)
    for attempt in range(3):
        try:
            request = Request(url, headers={"User-Agent": "PortfolioLab/1.0"})
            with urlopen(request, timeout=25) as response:
                content = json.load(response)
            observations = content["observations"]
            frame = pd.DataFrame(observations)
            frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
            prices = frame.set_index(pd.to_datetime(frame["date"]))[["value"]].dropna()
            prices.columns = ["DGS3MO"]
            if prices.empty:
                raise DataError("FRED returned no valid rates.")
            bundle = DataBundle(prices, {
                **query, "units": "percent per annum, investment basis",
                "retrieved_at_utc": utc_now(), "cache_hit": False,
                "vintage": "latest available; not a point-in-time ALFRED reconstruction",
            })
            _save_cache(cache_dir, query, bundle)
            return bundle
        except HTTPError as exc:
            if 400 <= exc.code < 500 and exc.code != 429:
                raise DataError(f"FRED request rejected (HTTP {exc.code}); check credentials.") from None
        except (URLError, TimeoutError, ValueError, KeyError):
            pass
        if attempt < 2:
            time.sleep(2 ** attempt)
    raise DataError("FRED request failed. No constant-rate fallback was applied.") from None


def risk_free_returns(index: pd.DatetimeIndex, annual_rate: float = .02,
                      yields_percent: pd.Series | None = None) -> tuple[pd.Series, dict]:
    """Cash-return proxy over each close-to-close calendar interval.

    The fixed rate is effective annual. For FRED, treating a quoted Treasury yield
    as effective annual is an approximation, not the realized return of a bill.
    A FRED quote must predate the START of the interval (strict backward as-of).
    """
    index = pd.DatetimeIndex(index)
    if len(index) < 2 or index.has_duplicates or not index.is_monotonic_increasing:
        raise DataError("Risk-free dates must be increasing, unique, and contain two observations.")
    days = np.diff(index.values).astype("timedelta64[D]").astype(float)
    if yields_percent is None:
        if not np.isfinite(annual_rate) or annual_rate <= -1:
            raise DataError("Invalid fixed annual risk-free assumption.")
        rates = np.full(len(days), annual_rate)
        metadata = {"source": "fixed assumption", "annual_effective_rate": annual_rate}
    else:
        quotes = yields_percent.dropna().sort_index()
        if quotes.empty or quotes.index.has_duplicates:
            raise DataError("FRED yields are empty or contain duplicate dates.")
        loc = quotes.index.searchsorted(index[:-1], side="left") - 1
        if (loc < 0).any():
            raise DataError("No risk-free quote predates the first return interval.")
        ages = (index[:-1] - quotes.index[loc]).days
        if (ages > 10).any():
            raise DataError("Risk-free observations are stale by more than 10 calendar days.")
        rates = quotes.iloc[loc].to_numpy(dtype=float) / 100
        if not np.isfinite(rates).all() or (rates <= -1).any():
            raise DataError("Risk-free annualized yields are invalid.")
        metadata = {"source": "FRED DGS3MO proxy", "max_quote_age_days": int(max(ages)),
                    "method": "strictly prior quote; effective-annual approximation; actual/365.25"}
    values = np.expm1(np.log1p(rates) * days / 365.25)
    return pd.Series(values, index=index[1:], name="risk_free_return"), metadata


In [ ]:
%%writefile portfolio_lab/engine.py
"""Self-financing holdings engine with costs, drift, and exact P&L attribution."""

from dataclasses import dataclass
import numpy as np
import pandas as pd


@dataclass
class Backtest:
    name: str
    nav: pd.Series
    returns: pd.Series
    weights: pd.DataFrame
    positions: pd.DataFrame
    pnl: pd.DataFrame
    contributions: pd.DataFrame
    ledger: pd.DataFrame


def rebalance_due(current: pd.Timestamp, previous: pd.Timestamp, frequency: str) -> bool:
    """Trade at the close of the first observed session in a new calendar period."""
    if frequency == "none":
        return False
    if frequency == "monthly":
        return (current.year, current.month) != (previous.year, previous.month)
    if frequency == "quarterly":
        return (current.year, current.quarter) != (previous.year, previous.quarter)
    if frequency == "annual":
        return current.year != previous.year
    raise ValueError(f"Unknown frequency: {frequency}")


def rebalance_with_cost(values: np.ndarray, target: np.ndarray,
                        cost_rate: float) -> tuple[np.ndarray, float, float]:
    """Solve V_after + c * sum(abs(w*V_after - holdings_before)) = V_before.

    Bisection is stable because 0 <= c < 1 and long-only target weights sum to one.
    Computing costs from pre-fee target weights would miss the extra selling
    necessary to fund those costs.
    """
    total = float(values.sum())
    if cost_rate == 0:
        after = total * target
        return after, 0.0, float(np.abs(after - values).sum())
    lower, upper = 0.0, total
    for _ in range(60):
        middle = (lower + upper) / 2
        balance = middle + cost_rate * np.abs(middle * target - values).sum() - total
        if balance > 0:
            upper = middle
        else:
            lower = middle
    after = ((lower + upper) / 2) * target
    traded = float(np.abs(after - values).sum())
    return after, float(total - after.sum()), traded


def backtest(prices: pd.DataFrame, target_weights: dict[str, float],
             initial_capital: float = 100_000, frequency: str = "monthly",
             cost_bps: float = 5.0, name: str = "Portfolio") -> Backtest:
    """Simulate an already-invested long-only portfolio, using total-return units.

    The first close is the endowed baseline; entry and final liquidation costs
    are excluded symmetrically. No external cash flows, leverage, tax, or cash
    sleeve. Today's return uses yesterday's holdings; a close rebalance affects
    the next interval. Adjusted units are not broker-executable share counts.
    """
    symbols = list(target_weights)
    if frequency not in {"monthly", "quarterly", "annual", "none"}:
        raise ValueError("Unsupported frequency.")
    if not np.isfinite(initial_capital) or initial_capital <= 0:
        raise ValueError("initial_capital must be finite and positive.")
    if not np.isfinite(cost_bps) or not 0 <= cost_bps <= 1000:
        raise ValueError("cost_bps must lie in [0, 1000].")
    target = np.array(list(target_weights.values()), dtype=float)
    if target.size == 0 or not np.isfinite(target).all() or (target < 0).any():
        raise ValueError("Weights must be finite and nonnegative.")
    if not np.isclose(target.sum(), 1.0, rtol=0, atol=1e-10):
        raise ValueError("Weights must sum to one.")
    if len(prices) < 2 or prices.index.has_duplicates or not prices.index.is_monotonic_increasing:
        raise ValueError("Need two or more unique, increasing price observations.")
    p = prices.loc[:, symbols].to_numpy(dtype=float)
    if not np.isfinite(p).all() or (p <= 0).any():
        raise ValueError("Prices must be finite, nonmissing, and positive.")
    # Price ratios are computed once; holdings must evolve sequentially.
    changes = p[1:] / p[:-1] - 1
    n, m = p.shape
    positions = np.zeros((n, m))
    pnl = np.zeros((n, m))
    contributions = np.zeros((n, m + 1))
    nav, fees, turnover, gross_traded = (np.zeros(n) for _ in range(4))
    rebalanced = np.zeros(n, dtype=bool)
    positions[0] = initial_capital * target
    nav[0] = initial_capital
    rate = cost_bps / 10_000
    for row in range(1, n):
        pnl[row] = positions[row - 1] * changes[row - 1]
        before = positions[row - 1] + pnl[row]
        contributions[row, :m] = pnl[row] / nav[row - 1]
        if rebalance_due(prices.index[row], prices.index[row - 1], frequency):
            after, fee, traded = rebalance_with_cost(before, target, rate)
            positions[row] = after
            fees[row], gross_traded[row] = fee, traded
            turnover[row] = traded / before.sum()  # Gross traded notional, not half-turnover.
            rebalanced[row] = True
        else:
            positions[row] = before
        nav[row] = positions[row].sum()
        contributions[row, -1] = -fees[row] / nav[row - 1]
    index = prices.index
    nav_series = pd.Series(nav, index=index, name=name)
    returns = nav_series.pct_change(fill_method=None).iloc[1:]
    contribution_frame = pd.DataFrame(
        contributions, index=index, columns=symbols + ["Trading costs"]
    )
    np.testing.assert_allclose(
        contribution_frame.iloc[1:].sum(axis=1), returns, atol=2e-12, rtol=1e-10,
        err_msg="Daily return attribution failed to reconcile."
    )
    total_pnl = float(pnl.sum() - fees.sum())
    np.testing.assert_allclose(total_pnl, nav[-1] - initial_capital, atol=1e-7, rtol=1e-10)
    ledger = pd.DataFrame({
        "nav": nav, "fees": fees, "gross_traded_notional": gross_traded,
        "gross_turnover": turnover, "rebalance": rebalanced,
    }, index=index)
    return Backtest(
        name=name, nav=nav_series, returns=returns,
        weights=pd.DataFrame(positions / nav[:, None], index=index, columns=symbols),
        positions=pd.DataFrame(positions, index=index, columns=symbols),
        pnl=pd.DataFrame(pnl, index=index, columns=symbols),
        contributions=contribution_frame, ledger=ledger,
    )


In [ ]:
%%writefile portfolio_lab/analytics.py
"""Return, tail-risk, attribution and scenario calculations with explicit conventions."""

import numpy as np
import pandas as pd
from .engine import Backtest


def safe_ratio(numerator: float, denominator: float) -> float:
    """Undefined ratios are NaN, never silently zero or infinite."""
    if not np.isfinite(denominator) or abs(denominator) < 1e-14:
        return float("nan")
    return float(numerator / denominator)


def drawdown(nav: pd.Series) -> pd.Series:
    return (nav / nav.cummax() - 1).rename("drawdown")


def drawdown_episodes(nav: pd.Series) -> pd.DataFrame:
    """Peak-to-recovery calendar durations; ongoing episodes have no recovery date."""
    episodes, peak_date, peak_value = [], nav.index[0], float(nav.iloc[0])
    trough_date, trough_value, active = peak_date, peak_value, False
    for date, value in nav.iloc[1:].items():
        value = float(value)
        if value >= peak_value * (1 - 1e-12):
            if active:
                episodes.append({"peak": peak_date, "trough": trough_date, "recovery": date,
                                 "depth": trough_value / peak_value - 1,
                                 "duration_days": (date - peak_date).days, "recovered": True})
            peak_date, peak_value = date, value
            trough_date, trough_value, active = date, value, False
        else:
            active = True
            if value < trough_value:
                trough_date, trough_value = date, value
    if active:
        episodes.append({"peak": peak_date, "trough": trough_date, "recovery": pd.NaT,
                         "depth": trough_value / peak_value - 1,
                         "duration_days": (nav.index[-1] - peak_date).days, "recovered": False})
    return pd.DataFrame(episodes, columns=[
        "peak", "trough", "recovery", "depth", "duration_days", "recovered"
    ]).sort_values("depth").reset_index(drop=True)


def periodic_returns(nav: pd.Series, frequency: str = "ME") -> pd.Series:
    """Compound from period-end NAV; include the initial partial period explicitly."""
    ends = nav.resample(frequency).last().dropna()
    beginning = ends.shift(1)
    beginning.iloc[0] = nav.iloc[0]
    return ends / beginning - 1


def monthly_heatmap(nav: pd.Series) -> pd.DataFrame:
    returns = periodic_returns(nav)
    frame = pd.DataFrame({"year": returns.index.year, "month": returns.index.month,
                          "return": returns.values})
    return frame.pivot(index="year", columns="month", values="return").reindex(columns=range(1, 13))


def summary_metrics(result: Backtest, benchmark: Backtest, risk_free: pd.Series,
                    annualization: int = 252, confidence: float = .95) -> dict[str, float]:
    """CAGR uses elapsed calendar time; daily moment estimators use 252 sessions/year."""
    returns = result.returns
    if not returns.index.equals(benchmark.returns.index) or not returns.index.equals(risk_free.index):
        raise ValueError("Portfolio, benchmark and risk-free returns must align exactly.")
    if len(returns) < 2 or not np.isfinite(risk_free.to_numpy()).all():
        raise ValueError("Metrics require finite, aligned return observations.")
    years = (result.nav.index[-1] - result.nav.index[0]).days / 365.25
    if years <= 0:
        raise ValueError("Performance interval has zero calendar length.")
    excess = returns - risk_free
    benchmark_excess = benchmark.returns - risk_free
    active = returns - benchmark.returns
    volatility = float(returns.std(ddof=1) * np.sqrt(annualization))
    excess_vol = float(excess.std(ddof=1) * np.sqrt(annualization))
    downside = float(np.sqrt(np.mean(np.minimum(excess.to_numpy(), 0) ** 2) * annualization))
    total_return = float(result.nav.iloc[-1] / result.nav.iloc[0] - 1)
    cagr = float(np.expm1(np.log1p(total_return) / years))
    dd = drawdown(result.nav)
    beta = safe_ratio(float(excess.cov(benchmark_excess)), float(benchmark_excess.var(ddof=1)))
    alpha = float((excess.mean() - beta * benchmark_excess.mean()) * annualization)
    tracking_error = float(active.std(ddof=1) * np.sqrt(annualization))
    losses = -returns.to_numpy()
    var = float(np.quantile(losses, confidence))
    es = float(losses[losses >= var].mean())
    episodes = drawdown_episodes(result.nav)
    return {
        "total_return": total_return, "cagr": cagr, "annual_volatility": volatility,
        "sharpe": safe_ratio(float(excess.mean() * annualization), excess_vol),
        "sortino": safe_ratio(float(excess.mean() * annualization), downside),
        "max_drawdown": float(dd.min()), "calmar": safe_ratio(cagr, abs(float(dd.min()))),
        "tracking_error": tracking_error,
        "information_ratio": safe_ratio(float(active.mean() * annualization), tracking_error),
        "beta": beta, "alpha_annual_arithmetic": alpha,
        "var_daily": var, "expected_shortfall_daily": es,
        "positive_day_fraction": float((returns > 0).mean()),
        "longest_drawdown_days": float(episodes.duration_days.max()) if len(episodes) else 0.,
        "total_fees": float(result.ledger.fees.sum()),
        "annual_gross_turnover": float(result.ledger.gross_turnover.sum() / years),
        "ending_nav": float(result.nav.iloc[-1]),
        "return_observations": float(len(returns)),
    }


def rolling_metrics(result: Backtest, benchmark: Backtest, risk_free: pd.Series,
                    window: int = 126, annualization: int = 252) -> pd.DataFrame:
    excess, b_excess = result.returns - risk_free, benchmark.returns - risk_free
    std = excess.rolling(window).std(ddof=1).replace(0, np.nan)
    b_var = b_excess.rolling(window).var(ddof=1).replace(0, np.nan)
    return pd.DataFrame({
        "annual_volatility": result.returns.rolling(window).std(ddof=1) * np.sqrt(annualization),
        "sharpe": excess.rolling(window).mean() / std * np.sqrt(annualization),
        "beta": excess.rolling(window).cov(b_excess) / b_var,
    })


def pnl_attribution(result: Backtest) -> pd.Series:
    """Dollar contributions add exactly to terminal NAV minus initial capital."""
    values = result.pnl.sum()
    values.loc["Trading costs"] = -result.ledger.fees.sum()
    np.testing.assert_allclose(values.sum(), result.nav.iloc[-1] - result.nav.iloc[0],
                               rtol=1e-10, atol=1e-7)
    return values.rename("pnl_usd")


def risk_contributions(asset_returns: pd.DataFrame, weights: pd.Series,
                       annualization: int = 252) -> pd.DataFrame:
    """Euler volatility allocation at CURRENT weights using full-sample covariance.

    This is an ex-post risk description, not a predictive model or a backtest input.
    Negative components can reflect hedging; they must not be clipped.
    """
    covariance = asset_returns.loc[:, weights.index].cov().to_numpy() * annualization
    w = weights.to_numpy()
    variance = float(w @ covariance @ w)
    sigma = np.sqrt(max(variance, 0))
    components = w * (covariance @ w) / sigma if sigma > 1e-14 else np.full(len(w), np.nan)
    fraction = components / sigma if sigma > 1e-14 else np.full(len(w), np.nan)
    return pd.DataFrame({"weight": w, "volatility_contribution": components,
                         "risk_fraction": fraction}, index=weights.index)


def stress_scenarios(weights: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    """Illustrative instantaneous shocks; no asserted probabilities or historical labels."""
    assumptions = {
        "Equity selloff": {"SPY": -.30, "VEA": -.32, "VWO": -.38, "AGG": .03,
                           "IEF": .06, "TIP": -.02, "GLD": .08, "VNQ": -.35},
        "Rates + inflation shock": {"SPY": -.15, "VEA": -.15, "VWO": -.18, "AGG": -.08,
                                    "IEF": -.12, "TIP": -.06, "GLD": .04, "VNQ": -.20},
        "Broad risk rally": {"SPY": .20, "VEA": .18, "VWO": .22, "AGG": .03,
                             "IEF": .02, "TIP": .04, "GLD": -.05, "VNQ": .20},
    }
    shocks = pd.DataFrame(assumptions).T
    missing = set(weights.index) - set(shocks.columns)
    if missing:
        # Custom assets must receive explicit shock assumptions rather than zero shocks.
        return pd.DataFrame(), pd.Series(dtype=float, name="scenario_return")
    shocks = shocks.loc[:, weights.index]
    return shocks, shocks.mul(weights, axis=1).sum(axis=1).rename("scenario_return")


In [ ]:
%%writefile portfolio_lab/charts.py
"""Consistent, publication-ready charts with no network or browser dependency."""

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from .analytics import drawdown, monthly_heatmap

NAVY, TEAL, BLUE, CORAL = "#142D4E", "#087F8C", "#7B91B4", "#D66A4B"
COLORS = {"Portfolio": TEAL, "Buy & hold": BLUE, "Benchmark": CORAL}
STYLE = {
    "figure.facecolor": "#FAFBFD", "axes.facecolor": "#FAFBFD",
    "axes.edgecolor": "#DCE3EA", "axes.labelcolor": "#566579",
    "text.color": NAVY, "xtick.color": "#566579", "ytick.color": "#566579",
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 13,
    "grid.alpha": .40, "grid.color": "#DCE3EA",
    "legend.frameon": False, "savefig.facecolor": "#FAFBFD",
}
MONEY = FuncFormatter(lambda value, _: "$" + f"{value / 1000:,.0f}k")


def _finish(ax, title: str, percent: bool = False):
    ax.set_title(title, loc="left", pad=14)
    ax.grid(axis="y", zorder=0)
    ax.set_axisbelow(True)
    if percent:
        ax.yaxis.set_major_formatter(PercentFormatter(1))


def _nav(ax, analysis):
    for name, result in analysis.results.items():
        ax.plot(result.nav.index, result.nav, label=name, color=COLORS[name],
                lw=2.2 if name == "Portfolio" else 1.35, alpha=.95)
    _finish(ax, "Portfolio value · net of rebalancing costs")
    ax.yaxis.set_major_formatter(MONEY)
    ax.legend(loc="upper left", ncol=3, fontsize=9)


def _drawdowns(ax, analysis):
    for name, result in analysis.results.items():
        values = drawdown(result.nav)
        ax.plot(values.index, values, color=COLORS[name], lw=1.15, label=name)
    _finish(ax, "Drawdowns from prior high", percent=True)


def _rolling_vol(ax, analysis):
    for name, frame in analysis.rolling.items():
        ax.plot(frame.index, frame.annual_volatility, color=COLORS[name], lw=1.25, label=name)
    _finish(ax, f"Rolling volatility · {analysis.config.rolling_window} sessions", percent=True)


def _attribution(ax, analysis):
    data = analysis.attribution.sort_values()
    ax.barh(data.index, data.values, color=[TEAL if v >= 0 else CORAL for v in data])
    ax.axvline(0, lw=.8, color=BLUE)
    ax.xaxis.set_major_formatter(MONEY)
    ax.set_title("P&L attribution · reconciled dollars", loc="left", pad=14)
    ax.grid(axis="x")
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", labelsize=9)


def create_figures(analysis) -> dict[str, plt.Figure]:
    """Create independent figures so both notebook and HTML exports reuse one calculation."""
    figures = {}
    label = "SIMULATED DATA • NOT MARKET PERFORMANCE" if analysis.metadata["data"]["synthetic"] else "HISTORICAL RESEARCH • USD"
    with plt.rc_context(STYLE):
        for key, drawing in [("growth", _nav), ("drawdowns", _drawdowns),
                             ("rolling_volatility", _rolling_vol)]:
            fig, ax = plt.subplots(figsize=(11, 4.3), layout="constrained")
            drawing(ax, analysis)
            fig.suptitle(label, x=.01, ha="left", fontsize=8, color=CORAL, fontweight="bold")
            figures[key] = fig

        fig, ax = plt.subplots(figsize=(10, 5.3), layout="constrained")
        matrix = monthly_heatmap(analysis.results["Portfolio"].nav)
        finite = matrix.to_numpy()[np.isfinite(matrix.to_numpy())]
        limit = max(float(np.abs(finite).max()), .01)
        image = ax.imshow(matrix, cmap="RdYlGn", vmin=-limit, vmax=limit, aspect="auto")
        for row in range(len(matrix)):
            for column in range(12):
                value = matrix.iloc[row, column]
                if np.isfinite(value):
                    ax.text(column, row, f"{value:.1%}", ha="center", va="center", fontsize=8,
                            color="white" if abs(value) > limit * .7 else NAVY)
        ax.set_xticks(range(12), ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
        ax.set_yticks(range(len(matrix)), matrix.index.astype(str))
        ax.set_title("Portfolio monthly returns · initial/final months may be partial", loc="left", pad=14)
        fig.colorbar(image, ax=ax, format=PercentFormatter(1), shrink=.8)
        figures["monthly_returns"] = fig

        fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
        correlation = analysis.prices.pct_change(fill_method=None).iloc[1:].corr()
        image = ax.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)
        for i in range(len(correlation)):
            for j in range(len(correlation)):
                value = correlation.iloc[i, j]
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if abs(value) > .6 else NAVY)
        ax.set_xticks(range(len(correlation)), correlation.columns)
        ax.set_yticks(range(len(correlation)), correlation.index)
        ax.set_title("Asset return correlations · full sample", loc="left", pad=14)
        fig.colorbar(image, ax=ax, shrink=.8)
        figures["correlation"] = fig

        fig, ax = plt.subplots(figsize=(11, 4.8), layout="constrained")
        weights = analysis.results["Portfolio"].weights
        palette = plt.get_cmap("tab20")(np.linspace(0, .85, len(weights.columns)))
        ax.stackplot(weights.index, weights.to_numpy().T, labels=weights.columns,
                     colors=palette, alpha=.90)
        _finish(ax, "Portfolio weights · drift and scheduled rebalancing", percent=True)
        ax.set_ylim(0, 1)
        ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(.5, -.13), fontsize=9)
        figures["weights"] = fig

        fig, ax = plt.subplots(figsize=(9, 4.8), layout="constrained")
        _attribution(ax, analysis)
        figures["pnl_attribution"] = fig

        fig, ax = plt.subplots(figsize=(9, 4.8), layout="constrained")
        risk = analysis.risk
        x = np.arange(len(risk))
        ax.bar(x - .18, risk.weight, .36, label="Capital weight", color=BLUE)
        ax.bar(x + .18, risk.risk_fraction, .36, label="Share of volatility", color=TEAL)
        ax.set_xticks(x, risk.index)
        _finish(ax, "Current allocation vs. full-sample risk contribution", percent=True)
        ax.legend(loc="upper right")
        figures["risk_contributions"] = fig

        fig, ax = plt.subplots(figsize=(10, 4.3), layout="constrained")
        if len(analysis.stress):
            ax.barh(analysis.stress.index, analysis.stress.values,
                    color=[TEAL if v >= 0 else CORAL for v in analysis.stress])
            ax.xaxis.set_major_formatter(PercentFormatter(1))
            ax.axvline(0, color=BLUE, lw=.8)
            ax.grid(axis="x")
        else:
            ax.text(.5, .5, "Define explicit shocks for your custom assets.",
                    ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Illustrative shocks · current holdings · no probabilities", loc="left", pad=14)
        figures["stress"] = fig

        fig = plt.figure(figsize=(14, 10), layout="constrained")
        grid = fig.add_gridspec(3, 2, height_ratios=[.40, 1, 1])
        banner = fig.add_subplot(grid[0, :])
        banner.axis("off")
        banner.text(0, .94, "PORTFOLIO LAB", fontsize=11, fontweight="bold", color=TEAL)
        banner.text(0, .53, "Multi-Asset Performance & Risk", fontsize=23, fontweight="bold")
        banner.text(0, .09, label, fontsize=9, color=CORAL, fontweight="bold")
        metrics = analysis.metrics.loc["Portfolio"]
        for position, (caption, value) in enumerate([
            ("CAGR", f"{metrics.cagr:.2%}"), ("VOLATILITY", f"{metrics.annual_volatility:.2%}"),
            ("SHARPE", f"{metrics.sharpe:.2f}"), ("MAX DRAWDOWN", f"{metrics.max_drawdown:.2%}")
        ]):
            x = .57 + position * .108
            banner.text(x, .61, caption, fontsize=8, color="#566579")
            banner.text(x, .22, value, fontsize=18, fontweight="bold")
        _nav(fig.add_subplot(grid[1, 0]), analysis)
        _drawdowns(fig.add_subplot(grid[1, 1]), analysis)
        _rolling_vol(fig.add_subplot(grid[2, 0]), analysis)
        _attribution(fig.add_subplot(grid[2, 1]), analysis)
        figures["executive_summary"] = fig
        for name, chart in figures.items():
            if name != "executive_summary":
                chart.suptitle(label, x=.01, ha="left", fontsize=8, color=CORAL, fontweight="bold")
    return figures


def save_figures(figures: dict, directory: Path) -> dict[str, Path]:
    directory.mkdir(parents=True, exist_ok=True)
    paths = {}
    for name, figure in figures.items():
        path = directory / f"{name}.png"
        figure.savefig(path, dpi=150, bbox_inches="tight")
        figure.savefig(directory / f"{name}.svg", bbox_inches="tight")
        paths[name] = path
    return paths


In [ ]:
%%writefile portfolio_lab/report.py
"""Self-contained HTML dashboard: embedded SVG charts, native controls, no CDN."""

from base64 import b64encode
from html import escape
from io import BytesIO
from pathlib import Path
import json
import numpy as np
from .data import atomic_write


def _image(figure, title: str) -> str:
    buffer = BytesIO()
    figure.savefig(buffer, format="svg", bbox_inches="tight")
    data = b64encode(buffer.getvalue()).decode()
    return f'<img src="data:image/svg+xml;base64,{data}" alt="{escape(title)}" loading="lazy">'


def _clean_number(value):
    return float(value) if np.isfinite(value) else None


def write_dashboard(analysis, figures: dict, path: Path):
    synthetic = analysis.metadata["data"]["synthetic"]
    badge = "SIMULATED DATA · DEMONSTRATION ONLY" if synthetic else "HISTORICAL RESEARCH · USD"
    first, last = analysis.prices.index[[0, -1]]
    rf = analysis.metadata["risk_free"]
    rf_label = (f"Fixed {rf['annual_effective_rate']:.2%} effective annual cash assumption"
                if rf["source"] == "fixed assumption" else "Lagged FRED DGS3MO yield proxy")
    payload = {
        "dates": analysis.prices.index.strftime("%Y-%m-%d").tolist(),
        "nav": {name: result.nav.round(8).tolist() for name, result in analysis.results.items()},
        "metrics": {name: {key: _clean_number(value) for key, value in row.items()}
                    for name, row in analysis.metrics.to_dict("index").items()},
    }
    encoded = json.dumps(payload, allow_nan=False).replace("<", "\\u003c")
    metric_titles = [
        ("cagr", "CAGR", "percent"), ("annual_volatility", "Annual volatility", "percent"),
        ("sharpe", "Sharpe ratio", "number"), ("max_drawdown", "Maximum drawdown", "percent"),
        ("ending_nav", "Ending value", "money"), ("total_fees", "Rebalancing costs", "money"),
    ]
    cards = "".join(
        f'<div class="kpi"><span>{title}</span><strong id="metric-{key}">—</strong>'
        f'<small>{kind}</small></div>' for key, title, kind in metric_titles
    )
    figures_html = {
        key: f'<article class="chart-card">{_image(fig, key.replace("_", " "))}</article>'
        for key, fig in figures.items() if key not in {"growth", "executive_summary"}
    }
    table = analysis.metrics.loc[:, ["cagr", "annual_volatility", "sharpe", "sortino",
                                    "max_drawdown", "beta", "tracking_error"]].copy()
    table.columns = ["CAGR", "Volatility", "Sharpe", "Sortino", "Max drawdown", "Beta", "Tracking error"]
    for col in ["CAGR", "Volatility", "Max drawdown", "Tracking error"]:
        table[col] = table[col].map(lambda x: f"{x:.2%}")
    for col in ["Sharpe", "Sortino", "Beta"]:
        table[col] = table[col].map(lambda x: f"{x:.2f}" if np.isfinite(x) else "N/A")
    comparison = table.to_html(classes="comparison", border=0)
    warnings = analysis.metadata["data_quality"]["warnings"]
    warning_html = "".join(f"<li>{escape(warning)}</li>" for warning in warnings)
    if not warning_html:
        warning_html = "<li>No flagged issues under the implemented quality checks.</li>"
    provenance = escape(json.dumps(analysis.metadata, indent=2))
    html = TEMPLATE
    replacements = {
        "__BADGE__": badge, "__FIRST__": str(first.date()), "__LAST__": str(last.date()),
        "__RF__": escape(rf_label), "__CARDS__": cards, "__PAYLOAD__": encoded,
        "__TABLE__": comparison, "__PROVENANCE__": provenance, "__WARNINGS__": warning_html,
        "__COST__": f"{analysis.config.cost_bps:g}",
        "__ROWS__": str(len(analysis.prices)), "__ASSETS__": str(len(analysis.prices.columns)),
        "__REBALANCE__": escape(analysis.config.rebalance),
        "__ANNUALIZATION__": str(analysis.config.annualization),
    }
    for key, value in figures_html.items():
        replacements["__" + key.upper() + "__"] = value
    for token, replacement in replacements.items():
        html = html.replace(token, replacement)
    atomic_write(path, html)


TEMPLATE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Portfolio Lab | Performance & Risk</title>
<style>
:root{--ink:#142d4e;--muted:#637086;--teal:#087f8c;--line:#dce3ea;--coral:#d66a4b}
*{box-sizing:border-box}body{margin:0;background:#f3f6fa;color:var(--ink);font:15px/1.5 system-ui,-apple-system,sans-serif}
.wrap{max-width:1420px;margin:auto;padding:30px 40px 60px}.brand{font-size:12px;letter-spacing:.2em;font-weight:800;color:var(--teal)}
.top{display:flex;justify-content:space-between;gap:20px;align-items:start}.badge{padding:7px 12px;background:#fff0e9;color:#9b4029;border:1px solid #f2d1c2;font-size:11px;font-weight:750;border-radius:24px}
h1{font-size:36px;line-height:1.15;margin:12px 0 9px;letter-spacing:-.04em}h2{font-size:20px;margin:0 0 5px}p{color:var(--muted);margin:6px 0 20px}
.controls{display:flex;gap:12px;align-items:center;flex-wrap:wrap;margin:24px 0 18px}select,button{font:inherit;border:1px solid var(--line);border-radius:8px;padding:9px 14px;background:white;color:var(--ink);cursor:pointer}
button:focus-visible,select:focus-visible{outline:3px solid #81c8d0;outline-offset:3px}
.tabs{margin-left:auto;display:flex;gap:7px;flex-wrap:wrap}.tabs button[aria-selected=true],.ranges button.active{background:var(--ink);color:white;border-color:var(--ink)}
.kpis{display:grid;grid-template-columns:repeat(6,1fr);gap:12px;margin:18px 0}.kpi{background:white;border:1px solid var(--line);padding:20px;border-radius:12px}
.kpi span{font-size:12px;color:var(--muted);display:block}.kpi strong{font-size:27px;display:block;margin:7px 0 0;letter-spacing:-.035em}.kpi small{display:none}
.panel{background:white;border:1px solid var(--line);border-radius:14px;padding:24px;margin-bottom:18px}.chart-head{display:flex;align-items:center;justify-content:space-between;gap:20px}.ranges{display:flex;gap:6px}.ranges button{padding:5px 10px;font-size:12px}
.chart-host{position:relative;width:100%;height:355px;margin-top:12px}svg.main-chart{width:100%;height:100%;overflow:visible;touch-action:pan-y}
.tooltip{display:none;position:absolute;pointer-events:none;z-index:10;background:var(--ink);color:white;padding:10px 14px;border-radius:8px;font-size:12px;white-space:nowrap;box-shadow:0 4px 18px #142d4e25}
.legend{display:flex;gap:20px;flex-wrap:wrap;font-size:12px;color:var(--muted)}.dot{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:5px}
.note{font-size:12px;color:var(--muted);margin:10px 0 0}.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}.chart-card{background:white;border:1px solid var(--line);border-radius:14px;overflow:hidden;padding:12px}.chart-card img{width:100%;height:auto;display:block}
.tab-content[hidden]{display:none}.table-wrap{overflow-x:auto}.comparison{width:100%;border-collapse:collapse;text-align:right;font-size:13px}.comparison th,.comparison td{padding:13px 10px;border-bottom:1px solid var(--line)}.comparison th:first-child{text-align:left}.comparison thead{color:var(--muted)}
pre{background:#f3f6fa;padding:18px;border-radius:8px;overflow:auto;max-height:480px;font-size:12px}li{margin:6px 0;color:var(--muted)}footer{margin-top:30px;font-size:12px;color:var(--muted)}
@media(max-width:1000px){.kpis{grid-template-columns:repeat(3,1fr)}.wrap{padding:24px}.tabs{margin-left:0}.top{display:block}.badge{display:inline-block;margin-top:12px}}
@media(max-width:680px){.grid{grid-template-columns:1fr}.kpis{grid-template-columns:repeat(2,1fr)}h1{font-size:29px}.wrap{padding:16px}.kpi{padding:14px}.kpi strong{font-size:24px}.panel{padding:16px}.chart-host{height:300px}.chart-head{display:block}.ranges{margin-top:10px}}
@media print{button,select,.controls{display:none}.tab-content[hidden]{display:block}.wrap{padding:0}.chart-card,.panel{break-inside:avoid}}
</style></head><body><main class="wrap">
<div class="top"><div><div class="brand">PORTFOLIO LAB / RESEARCH SYSTEMS</div>
<h1>Multi-Asset Performance & Risk</h1><p>__FIRST__ — __LAST__ · USD · __ASSETS__ assets · __ROWS__ price observations</p></div>
<span class="badge">__BADGE__</span></div>
<div class="controls"><label for="strategy">Inspect strategy</label><select id="strategy">
<option>Portfolio</option><option>Buy &amp; hold</option><option>Benchmark</option></select>
<div class="tabs" role="tablist" aria-label="Dashboard sections">
<button role="tab" aria-selected="true" data-tab="overview" aria-controls="overview">Overview</button>
<button role="tab" aria-selected="false" data-tab="risk" aria-controls="risk">Risk</button>
<button role="tab" aria-selected="false" data-tab="positions" aria-controls="positions">Attribution</button>
<button role="tab" aria-selected="false" data-tab="data" aria-controls="data">Data & assumptions</button></div></div>
<div class="kpis">__CARDS__</div>
<p class="note" id="metric-scope">All metrics use the full sample. Chart range buttons change the chart window only.</p>
<section id="overview" role="tabpanel" class="tab-content">
<div class="panel"><div class="chart-head"><div><h2>Portfolio value</h2><p class="note">Net of rebalancing costs · hover for exact observations</p></div>
<div class="ranges"><button data-years="1">1Y</button><button data-years="3">3Y</button><button data-years="0" class="active">All</button></div></div>
<div class="chart-host" id="chart-host"><svg class="main-chart" id="nav-chart" role="img" aria-label="Portfolio, buy-and-hold and benchmark values over time"></svg><div class="tooltip" id="tooltip"></div></div>
<div class="legend"><span><i class="dot" style="background:#087f8c"></i>Portfolio</span><span><i class="dot" style="background:#7b91b4"></i>Buy &amp; hold</span><span><i class="dot" style="background:#d66a4b"></i>Benchmark</span></div></div>
<div class="panel"><h2>Full-sample comparison</h2><p class="note">Same common history, starting capital and initial-holdings convention.</p><div class="table-wrap">__TABLE__</div></div>
<div class="grid">__DRAWDOWNS____MONTHLY_RETURNS__</div></section>
<section id="risk" role="tabpanel" class="tab-content" hidden>
<p>Rolling comparisons include all strategies. Asset correlations, component risk and scenarios describe the configured Portfolio.</p>
<div class="grid">__ROLLING_VOLATILITY____CORRELATION____RISK_CONTRIBUTIONS____STRESS__</div></section>
<section id="positions" role="tabpanel" class="tab-content" hidden><p>Portfolio allocation and dollar P&amp;L. Asset P&amp;L less trading costs equals the change in net asset value.</p>
<div class="grid">__WEIGHTS____PNL_ATTRIBUTION__</div></section>
<section id="data" role="tabpanel" class="tab-content" hidden>
<div class="panel"><h2>Model conventions</h2><ul>
<li>Risk-free input: __RF__. This is a cash-return proxy used for excess-return metrics.</li>
<li>Portfolio rebalancing: __REBALANCE__; benchmark: monthly; trades occur at the first observed session's close in the new period.</li>
<li>Cost: __COST__ basis points per dollar bought or sold. The cost is paid from portfolio value.</li>
<li>The portfolio begins already invested. Initial purchase and terminal liquidation costs are excluded for all strategies.</li>
<li>Adjusted total-return units include provider adjustments. Dividends are not added again.</li>
<li>Missing prices are never forward-filled. Only common observed sessions are used.</li>
<li>CAGR uses elapsed calendar years; volatility and moment-based ratios use __ANNUALIZATION__ sessions per year.</li>
<li>Universe selection is fixed. No taxes, leverage, external cash flows, or live order execution.</li>
</ul></div><div class="panel"><h2>Data-quality findings</h2><ul>__WARNINGS__</ul></div>
<div class="panel"><h2>Run provenance</h2><pre>__PROVENANCE__</pre></div></section>
<footer>Portfolio Lab · Reproducible quantitative research. Simulated runs are demonstrations; historical results describe the supplied data and stated assumptions.</footer>
</main><script id="portfolio-data" type="application/json">__PAYLOAD__</script>
<script>
"use strict";
const data=JSON.parse(document.getElementById("portfolio-data").textContent);
const colors={"Portfolio":"#087f8c","Buy & hold":"#7b91b4","Benchmark":"#d66a4b"};
const strategies=Object.keys(data.nav), svg=document.getElementById("nav-chart");
let selected="Portfolio",years=0, geometry=null;
const ns="http://www.w3.org/2000/svg";
function add(tag,attrs,text){const el=document.createElementNS(ns,tag);Object.entries(attrs).forEach(([k,v])=>el.setAttribute(k,v));if(text!==undefined)el.textContent=text;svg.appendChild(el);return el}
function metrics(){const values=data.metrics[selected];Object.entries(values).forEach(([key,value])=>{const node=document.getElementById("metric-"+key);if(!node)return;
let output="N/A";if(value!==null){output=["ending_nav","total_fees"].includes(key)?new Intl.NumberFormat("en-US",{style:"currency",currency:"USD",maximumFractionDigits:0}).format(value):key==="sharpe"?value.toFixed(2):(value*100).toFixed(2)+"%"}node.textContent=output});
document.getElementById("metric-scope").textContent=selected+" · full-sample metrics. Range buttons affect the chart only."}
function render(){if(document.getElementById("overview").hidden)return;svg.replaceChildren();const width=Math.max(svg.clientWidth,280),height=svg.clientHeight||355;
svg.setAttribute("viewBox","0 0 "+width+" "+height);const left=65,right=18,top=18,bottom=35;
const cutoff=years?new Date(data.dates.at(-1)+"T00:00:00Z").getTime()-years*365.25*86400000:0;
let start=years?data.dates.findIndex(d=>new Date(d+"T00:00:00Z").getTime()>=cutoff):0;if(start<0)start=0;
const last=data.dates.length-1;let low=Infinity,high=-Infinity;strategies.forEach(s=>data.nav[s].slice(start).forEach(v=>{low=Math.min(low,v);high=Math.max(high,v)}));
const pad=Math.max((high-low)*.12,high*.015);low-=pad;high+=pad;
const X=i=>left+(i-start)/Math.max(last-start,1)*(width-left-right),Y=v=>top+(high-v)/(high-low)*(height-top-bottom);
for(let j=0;j<5;j++){const v=low+(high-low)*j/4,y=Y(v);add("line",{x1:left,x2:width-right,y1:y,y2:y,stroke:"#e5eaf0"});add("text",{x:left-10,y:y+4,"text-anchor":"end",fill:"#637086","font-size":11},"$"+(v/1000).toFixed(0)+"k")}
for(let j=0;j<5;j++){const i=Math.round(start+(last-start)*j/4);add("text",{x:X(i),y:height-8,"text-anchor":"middle",fill:"#637086","font-size":11},data.dates[i].slice(0,7))}
strategies.filter(s=>s!==selected).concat([selected]).forEach(s=>{let path="";for(let i=start;i<=last;i++)path+=(i===start?"M":"L")+X(i).toFixed(2)+","+Y(data.nav[s][i]).toFixed(2);
add("path",{d:path,fill:"none",stroke:colors[s],"stroke-width":s===selected?2.6:1.4,opacity:s===selected?1:.65})});
const cursor=add("line",{x1:0,x2:0,y1:top,y2:height-bottom,stroke:"#9aaabd","stroke-dasharray":"4 4",visibility:"hidden"});
geometry={width,height,left,right,start,last,X,Y,cursor};}
svg.addEventListener("pointermove",event=>{if(!geometry)return;const g=geometry,box=svg.getBoundingClientRect(),px=event.clientX-box.left;
const i=Math.max(g.start,Math.min(g.last,Math.round(g.start+(px-g.left)/(g.width-g.left-g.right)*(g.last-g.start))));
g.cursor.setAttribute("x1",g.X(i));g.cursor.setAttribute("x2",g.X(i));g.cursor.setAttribute("visibility","visible");
const tip=document.getElementById("tooltip");tip.replaceChildren();const title=document.createElement("strong");title.textContent=data.dates[i];tip.appendChild(title);
strategies.forEach(s=>{const line=document.createElement("div");line.textContent=s+": "+new Intl.NumberFormat("en-US",{style:"currency",currency:"USD",maximumFractionDigits:0}).format(data.nav[s][i]);tip.appendChild(line)});
tip.style.display="block";tip.style.left=Math.max(0,Math.min(px+14,g.width-tip.offsetWidth-8))+"px";tip.style.top="18px"});
svg.addEventListener("pointerleave",()=>{document.getElementById("tooltip").style.display="none";if(geometry)geometry.cursor.setAttribute("visibility","hidden")});
document.getElementById("strategy").addEventListener("change",event=>{selected=event.target.value;metrics();render()});
document.querySelectorAll(".ranges button").forEach(button=>button.addEventListener("click",()=>{years=Number(button.dataset.years);document.querySelectorAll(".ranges button").forEach(b=>b.classList.toggle("active",b===button));render()}));
document.querySelectorAll("[data-tab]").forEach(button=>button.addEventListener("click",()=>{document.querySelectorAll("[data-tab]").forEach(b=>b.setAttribute("aria-selected",b===button?"true":"false"));document.querySelectorAll(".tab-content").forEach(p=>p.hidden=p.id!==button.dataset.tab);render()}));
window.addEventListener("resize",render);metrics();render();
</script></body></html>
"""


In [ ]:
%%writefile portfolio_lab/pipeline.py
"""Orchestration and reproducible exports; no data-provider calls inside analytics."""

from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import json
import os
import platform

import numpy as np
import pandas as pd

from .analytics import (monthly_heatmap, pnl_attribution, risk_contributions,
                        rolling_metrics, stress_scenarios, summary_metrics, drawdown_episodes)
from .config import Config
from .data import (DataError, atomic_write, clean_prices, fetch_fred_yields, fetch_yahoo,
                   generate_demo, load_csv, risk_free_returns, utc_now)
from .engine import Backtest, backtest


@dataclass
class Analysis:
    config: Config
    prices: pd.DataFrame
    risk_free: pd.Series
    results: dict[str, Backtest]
    metrics: pd.DataFrame
    rolling: dict[str, pd.DataFrame]
    risk: pd.DataFrame
    attribution: pd.Series
    shocks: pd.DataFrame
    stress: pd.Series
    metadata: dict


def run_analysis(config: Config, mode: str = "demo", cache_dir: Path | str = "data/cache",
                 csv_path: Path | str | None = None, use_fred: bool = False,
                 fred_api_key: str | None = None, refresh: bool = False) -> Analysis:
    """Build three comparable portfolios on an identical common price calendar."""
    cache_dir = Path(cache_dir)
    if mode == "demo":
        bundle = generate_demo(config)
    elif mode == "yahoo":
        bundle = fetch_yahoo(config, cache_dir, refresh)
    elif mode == "csv" and csv_path is not None:
        bundle = load_csv(Path(csv_path))
    else:
        raise DataError("mode must be demo, yahoo, or csv with a supplied csv_path.")
    prices, audit = clean_prices(bundle.prices, config)
    fred_metadata = None
    if use_fred:
        fred = fetch_fred_yields(config, fred_api_key or os.getenv("FRED_API_KEY", ""),
                                cache_dir, refresh)
        rf, rf_metadata = risk_free_returns(prices.index, yields_percent=fred.prices["DGS3MO"])
        fred_metadata = fred.metadata
    else:
        rf, rf_metadata = risk_free_returns(prices.index, config.fixed_rf_annual)
    results = {
        "Portfolio": backtest(prices, config.weights, config.initial_capital,
                              config.rebalance, config.cost_bps, "Portfolio"),
        "Buy & hold": backtest(prices, config.weights, config.initial_capital,
                               "none", config.cost_bps, "Buy & hold"),
        "Benchmark": backtest(prices, config.benchmark_weights, config.initial_capital,
                              "monthly", config.cost_bps, "Benchmark"),
    }
    benchmark = results["Benchmark"]
    metrics = pd.DataFrame({
        name: summary_metrics(result, benchmark, rf, config.annualization, config.confidence)
        for name, result in results.items()
    }).T
    rolling = {
        name: rolling_metrics(result, benchmark, rf, config.rolling_window, config.annualization)
        for name, result in results.items()
    }
    asset_returns = prices.pct_change(fill_method=None).iloc[1:]
    final_weights = results["Portfolio"].weights.iloc[-1]
    risk = risk_contributions(asset_returns, final_weights, config.annualization)
    shocks, stress = stress_scenarios(final_weights)
    metadata = {
        "generated_at_utc": utc_now(), "mode": mode, "data": bundle.metadata,
        "data_quality": audit, "risk_free": rf_metadata, "fred_download": fred_metadata,
        "assumptions": {
            "base_currency": "USD", "long_only": True, "fractional_total_return_units": True,
            "start": "Already invested at first close; entry and liquidation fees excluded.",
            "rebalance": "First observed session of the new period, after its return.",
            "costs": "Basis points on gross bought plus sold notional, funded from NAV.",
            "cash_flows": "None; no leverage, borrowing, tax, or standalone cash sleeve.",
            "benchmark": "Configured benchmark weights, monthly close rebalancing, same cost rate.",
            "covariance": "Full-sample ex-post estimate for descriptive risk attribution.",
        },
    }
    return Analysis(config, prices, rf, results, metrics, rolling, risk,
                    pnl_attribution(results["Portfolio"]), shocks, stress, metadata)


def export_analysis(analysis: Analysis, output_dir: Path | str = "reports") -> dict[str, Path]:
    """Export reusable numeric outputs, provenance, charts, and an offline HTML report."""
    from .charts import create_figures, save_figures
    from .report import write_dashboard
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    frames = {
        "cleaned_prices": analysis.prices, "metrics": analysis.metrics,
        "risk_free_returns": analysis.risk_free, "risk_contributions": analysis.risk,
        "pnl_attribution": analysis.attribution, "scenario_assumptions": analysis.shocks,
        "scenario_results": analysis.stress,
    }
    for name, result in analysis.results.items():
        slug = name.lower().replace(" & ", "_").replace(" ", "_")
        frames.update({
            f"{slug}_ledger": result.ledger, f"{slug}_positions": result.positions,
            f"{slug}_weights": result.weights, f"{slug}_daily_returns": result.returns,
            f"{slug}_return_contributions": result.contributions,
            f"{slug}_monthly_returns": monthly_heatmap(result.nav),
            f"{slug}_drawdowns": drawdown_episodes(result.nav),
            f"{slug}_rolling": analysis.rolling[name],
        })
    paths = {}
    for name, frame in frames.items():
        path = output_dir / f"{name}.csv"
        frame.to_csv(path, index=True, float_format="%.12g")
        paths[name] = path
    versions = {"python": platform.python_version()}
    for package in ("numpy", "pandas", "matplotlib", "yfinance"):
        try:
            versions[package] = version(package)
        except PackageNotFoundError:
            versions[package] = "not installed"
    manifest = {**analysis.metadata, "configuration": analysis.config.to_dict(),
                "environment": versions,
                "file_hashes": {p.name: sha256(p.read_bytes()).hexdigest() for p in paths.values()}}
    manifest_path = output_dir / "manifest.json"
    atomic_write(manifest_path, json.dumps(manifest, indent=2))
    paths["manifest"] = manifest_path
    figures = create_figures(analysis)
    paths.update(save_figures(figures, output_dir / "charts"))
    dashboard = output_dir / "dashboard.html"
    write_dashboard(analysis, figures, dashboard)
    paths["dashboard"] = dashboard
    import matplotlib.pyplot as plt
    for figure in figures.values():
        plt.close(figure)
    return paths


In [ ]:
%%writefile portfolio_lab/__main__.py
"""Command-line entry point: python -m portfolio_lab --mode demo."""

import argparse
import logging
from pathlib import Path
from .config import Config
from .data import DataError
from .pipeline import export_analysis, run_analysis


def main():
    parser = argparse.ArgumentParser(description="Multi-asset performance and risk dashboard")
    parser.add_argument("--mode", choices=["demo", "yahoo", "csv"], default="demo")
    parser.add_argument("--csv", type=Path, help="Adjusted-price CSV for --mode csv")
    parser.add_argument("--start", default="2015-01-01")
    parser.add_argument("--end", default="2026-01-01", help="Exclusive end date")
    parser.add_argument("--rebalance", choices=["monthly", "quarterly", "annual", "none"], default="monthly")
    parser.add_argument("--cost-bps", type=float, default=5)
    parser.add_argument("--fixed-rf", type=float, default=.02, help="Effective annual decimal rate")
    parser.add_argument("--fred", action="store_true", help="Use FRED_API_KEY from environment")
    parser.add_argument("--refresh", action="store_true")
    parser.add_argument("--output", type=Path, default=Path("reports"))
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    try:
        config = Config(start=args.start, end=args.end, rebalance=args.rebalance,
                        cost_bps=args.cost_bps, fixed_rf_annual=args.fixed_rf)
        analysis = run_analysis(config, mode=args.mode, csv_path=args.csv,
                                use_fred=args.fred, refresh=args.refresh)
        paths = export_analysis(analysis, args.output)
    except (DataError, ValueError, OSError) as exc:
        parser.exit(2, f"Error: {exc}\n")
    print(analysis.metrics[["cagr", "annual_volatility", "sharpe", "max_drawdown"]].round(4))
    print(f"Data source: {analysis.metadata['data']['source']}")
    print(f"Dashboard: {paths['dashboard'].resolve()}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/test_finance.py
"""Economic invariants and independent hand calculations; no market API calls."""
import unittest
import numpy as np
import pandas as pd
from portfolio_lab.analytics import (drawdown, drawdown_episodes, periodic_returns,
                                     pnl_attribution, risk_contributions, summary_metrics)
from portfolio_lab.data import risk_free_returns
from portfolio_lab.engine import backtest, rebalance_with_cost


def prices(values, dates=None):
    return pd.DataFrame(values, index=pd.to_datetime(dates) if dates else
                        pd.bdate_range("2024-01-02", periods=len(next(iter(values.values())))))


class HoldingsTests(unittest.TestCase):
    def test_buy_and_hold_matches_fixed_units(self):
        p = prices({"A": [100, 200, 100], "B": [100, 100, 100]})
        result = backtest(p, {"A": .5, "B": .5}, 1000, "none", 0)
        np.testing.assert_allclose(result.nav, [1000, 1500, 1000])
        self.assertAlmostEqual(result.weights.iloc[1]["A"], 2 / 3)
        self.assertAlmostEqual(result.returns.iloc[1], -1 / 3)

    def test_close_trade_cannot_change_same_day_pnl(self):
        p = prices({"A": [100, 200, 200], "B": [100, 100, 200]},
                   ["2024-01-31", "2024-02-01", "2024-02-02"])
        result = backtest(p, {"A": .5, "B": .5}, 1000, "monthly", 0)
        np.testing.assert_allclose(result.nav, [1000, 1500, 2250])
        self.assertTrue(result.ledger.rebalance.iloc[1])
        np.testing.assert_allclose(result.weights.iloc[1], [.5, .5])

    def test_cost_solver_matches_two_asset_closed_form(self):
        after, fee, traded = rebalance_with_cost(np.array([600., 400.]), np.array([.5, .5]), .01)
        np.testing.assert_allclose(after, [499., 499.], atol=1e-10)
        self.assertAlmostEqual(fee, 2)
        self.assertAlmostEqual(traded, 200)

    def test_cost_funding_full_rotation(self):
        after, fee, traded = rebalance_with_cost(np.array([1000., 0.]), np.array([0., 1.]), .01)
        expected = 1000 * (1 - .01) / (1 + .01)
        self.assertAlmostEqual(after.sum(), expected)
        self.assertAlmostEqual(fee, .01 * traded)
        self.assertAlmostEqual(after.sum() + fee, 1000)

    def test_reconciles_every_interval_with_costs(self):
        rng = np.random.default_rng(7)
        p = pd.DataFrame(100 * np.exp(np.cumsum(rng.normal(0, .02, (300, 3)), axis=0)),
                         index=pd.bdate_range("2023-01-02", periods=300), columns=list("ABC"))
        result = backtest(p, {"A": .6, "B": .3, "C": .1}, 100_000, "monthly", 20)
        np.testing.assert_allclose(result.contributions.iloc[1:].sum(axis=1), result.returns, atol=1e-12)
        self.assertAlmostEqual(pnl_attribution(result).sum(), result.nav.iloc[-1] - 100_000, places=6)
        np.testing.assert_allclose(result.positions.sum(axis=1), result.nav)
        np.testing.assert_allclose(result.weights.sum(axis=1), 1)
        np.testing.assert_allclose(result.ledger.fees, .002 * result.ledger.gross_traded_notional, atol=1e-8)

    def test_future_changes_do_not_change_prior_nav(self):
        p = prices({"A": [100, 102, 103, 104], "B": [100, 99, 101, 105]})
        first = backtest(p, {"A": .5, "B": .5}, frequency="monthly")
        modified = p.copy()
        modified.iloc[-1] *= 2
        second = backtest(modified, {"A": .5, "B": .5}, frequency="monthly")
        np.testing.assert_allclose(first.nav.iloc[:-1], second.nav.iloc[:-1])

    def test_constant_prices_have_no_trades_or_fees(self):
        p = pd.DataFrame(100., index=pd.bdate_range("2024-01-01", periods=100), columns=["A", "B"])
        result = backtest(p, {"A": .6, "B": .4}, cost_bps=25)
        np.testing.assert_allclose(result.nav, 100_000, atol=1e-8)
        self.assertLess(abs(result.ledger.fees.sum()), 1e-8)

    def test_rejects_leverage_or_negative_weights(self):
        p = prices({"A": [100, 101, 99], "B": [100, 102, 98]})
        for weights in [{"A": 1.1, "B": -.1}, {"A": .4, "B": .4}]:
            with self.assertRaises(ValueError):
                backtest(p, weights)


class AnalyticsTests(unittest.TestCase):
    def test_drawdown_and_recovery(self):
        nav = pd.Series([100, 120, 90, 100, 120, 110],
                        index=pd.date_range("2024-01-01", periods=6))
        self.assertAlmostEqual(drawdown(nav).min(), -.25)
        episodes = drawdown_episodes(nav)
        self.assertEqual(len(episodes), 2)
        self.assertTrue(episodes.iloc[0].recovered)
        self.assertEqual(episodes.iloc[0].duration_days, 3)
        self.assertFalse(episodes.iloc[1].recovered)

    def test_monthly_returns_compound_to_total(self):
        nav = pd.Series([100, 110, 121, 133.1], index=pd.to_datetime([
            "2024-01-15", "2024-01-31", "2024-02-29", "2024-03-15"]))
        monthly = periodic_returns(nav)
        np.testing.assert_allclose(monthly, [.1, .1, .1])
        self.assertAlmostEqual((1 + monthly).prod(), nav.iloc[-1] / nav.iloc[0])

    def test_identical_benchmark_has_beta_one_and_zero_tracking_error(self):
        p = prices({"A": [100, 102, 101, 103, 99, 104]})
        result = backtest(p, {"A": 1}, frequency="none", cost_bps=0)
        rf, _ = risk_free_returns(p.index, 0)
        metrics = summary_metrics(result, result, rf)
        self.assertAlmostEqual(metrics["beta"], 1)
        self.assertAlmostEqual(metrics["tracking_error"], 0)
        self.assertAlmostEqual(metrics["alpha_annual_arithmetic"], 0)
        self.assertTrue(np.isnan(metrics["information_ratio"]))

    def test_constant_returns_do_not_produce_infinite_sharpe(self):
        p = prices({"A": [100, 100, 100, 100]})
        result = backtest(p, {"A": 1})
        rf, _ = risk_free_returns(p.index, 0)
        metrics = summary_metrics(result, result, rf)
        self.assertTrue(np.isnan(metrics["sharpe"]))
        self.assertTrue(np.isnan(metrics["sortino"]))
        self.assertTrue(np.isnan(metrics["calmar"]))

    def test_tail_metrics_have_correct_loss_sign(self):
        p = prices({"A": [100, 90, 99, 89.1, 93.555]})
        result = backtest(p, {"A": 1}, frequency="none", cost_bps=0)
        rf, _ = risk_free_returns(p.index, 0)
        metrics = summary_metrics(result, result, rf)
        self.assertAlmostEqual(metrics["var_daily"], .1)
        self.assertAlmostEqual(metrics["expected_shortfall_daily"], .1)

    def test_cagr_uses_calendar_years(self):
        p = prices({"A": [100, 105, 110]}, ["2023-01-01", "2023-07-01", "2024-01-01"])
        result = backtest(p, {"A": 1}, frequency="none")
        rf, _ = risk_free_returns(p.index, 0)
        metrics = summary_metrics(result, result, rf)
        self.assertAlmostEqual(metrics["cagr"], 1.1 ** (365.25 / 365) - 1)

    def test_sortino_uses_all_observations_in_downside_rms(self):
        p = prices({"A": [100, 110, 99, 108.9, 98.01]})
        result = backtest(p, {"A": 1}, frequency="none")
        rf = pd.Series(.01, index=result.returns.index)
        excess = np.array([.09, -.11, .09, -.11])
        expected = excess.mean() * np.sqrt(252) / np.sqrt(np.mean(np.minimum(excess, 0) ** 2))
        self.assertAlmostEqual(summary_metrics(result, result, rf)["sortino"], expected)

    def test_risk_contributions_sum_to_volatility(self):
        returns = pd.DataFrame({"A": [.02, -.01, .03, -.03], "B": [.01, .01, -.01, -.02]})
        w = pd.Series({"A": .6, "B": .4})
        result = risk_contributions(returns, w)
        expected = np.sqrt(w.values @ (returns.cov().values * 252) @ w.values)
        self.assertAlmostEqual(result.volatility_contribution.sum(), expected)
        self.assertAlmostEqual(result.risk_fraction.sum(), 1)


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%writefile tests/test_data.py
"""Data boundaries, provider schemas, caching, and information timing."""
from pathlib import Path
from tempfile import TemporaryDirectory
import unittest
from unittest.mock import patch
import numpy as np
import pandas as pd
from portfolio_lab.config import Config
from portfolio_lab.data import (DataError, clean_prices, extract_adjusted_close,
    fetch_yahoo, generate_demo, risk_free_returns, fetch_fred_yields)
from portfolio_lab.pipeline import run_analysis


def small_config():
    return Config(start="2024-01-01", end="2024-02-01",
                  weights={"A": .5, "B": .5}, benchmark_weights={"A": 1},
                  min_observations=3, rolling_window=3)


class DataTests(unittest.TestCase):
    def setUp(self):
        self.config = small_config()
        self.p = pd.DataFrame({"A": [100, 101, 102, 103], "B": [100, 99, 101, 102]},
                              index=pd.bdate_range("2024-01-02", periods=4))

    def test_missing_interior_price_fails(self):
        self.p.loc[self.p.index[1], "A"] = np.nan
        with self.assertRaisesRegex(DataError, "Interior"):
            clean_prices(self.p, self.config)

    def test_common_inception_is_trimmed_and_reported(self):
        self.p.loc[self.p.index[0], "A"] = np.nan
        clean, audit = clean_prices(self.p, self.config)
        self.assertEqual(len(clean), 3)
        self.assertEqual(audit["trimmed_rows"], 1)

    def test_duplicate_dates_fail(self):
        with self.assertRaisesRegex(DataError, "Duplicate"):
            clean_prices(pd.concat([self.p, self.p.iloc[[-1]]]), self.config)

    def test_nonpositive_prices_fail(self):
        self.p.iloc[1, 0] = 0
        with self.assertRaisesRegex(DataError, "positive"):
            clean_prices(self.p, self.config)

    def test_infinite_prices_fail(self):
        p = self.p.astype(float)
        p.iloc[1, 0] = np.inf
        with self.assertRaises(DataError):
            clean_prices(p, self.config)

    def test_yahoo_multiindex_field_first(self):
        raw = pd.concat({"Adj Close": self.p, "Close": self.p * 2}, axis=1)
        pd.testing.assert_frame_equal(extract_adjusted_close(raw, ["A", "B"]), self.p)

    def test_yahoo_multiindex_ticker_first(self):
        raw = pd.concat({"Adj Close": self.p, "Close": self.p * 2}, axis=1).swaplevel(axis=1)
        pd.testing.assert_frame_equal(extract_adjusted_close(raw, ["A", "B"]), self.p)

    def test_yahoo_single_ticker_flat_schema(self):
        raw = pd.DataFrame({"Adj Close": [100, 101]}, index=self.p.index[:2])
        self.assertEqual(extract_adjusted_close(raw, ["A"]).columns.tolist(), ["A"])

    def test_raw_close_is_never_silently_substituted(self):
        with self.assertRaisesRegex(DataError, "Adjusted"):
            extract_adjusted_close(pd.concat({"Close": self.p}, axis=1), ["A", "B"])

    def test_cache_round_trip_does_not_call_provider_twice(self):
        calls = []
        def download(*args, **kwargs):
            calls.append(kwargs)
            return pd.concat({"Adj Close": self.p}, axis=1)
        with TemporaryDirectory() as directory:
            first = fetch_yahoo(self.config, Path(directory), downloader=download)
            second = fetch_yahoo(self.config, Path(directory), downloader=download)
        self.assertEqual(len(calls), 1)
        self.assertFalse(calls[0]["auto_adjust"])
        self.assertTrue(second.metadata["cache_hit"])
        np.testing.assert_allclose(first.prices, second.prices)

    def test_provider_failure_does_not_become_demo_data(self):
        def broken(*args, **kwargs):
            raise ConnectionError("network unavailable")
        with TemporaryDirectory() as directory, patch("portfolio_lab.data.time.sleep"):
            with self.assertRaisesRegex(DataError, "failed after"):
                fetch_yahoo(self.config, Path(directory), downloader=broken)

    def test_constant_risk_free_includes_weekend_accrual(self):
        dates = pd.to_datetime(["2024-01-05", "2024-01-08", "2024-01-09"])
        rf, _ = risk_free_returns(dates, .05)
        np.testing.assert_allclose(rf, [(1.05) ** (3 / 365.25) - 1, (1.05) ** (1 / 365.25) - 1])

    def test_risk_free_uses_quote_strictly_before_interval_start(self):
        dates = pd.to_datetime(["2024-01-05", "2024-01-08", "2024-01-09"])
        yields = pd.Series([2., 20., 50.],
                           index=pd.to_datetime(["2024-01-04", "2024-01-05", "2024-01-08"]))
        rf, _ = risk_free_returns(dates, yields_percent=yields)
        np.testing.assert_allclose(rf, [(1.02) ** (3 / 365.25) - 1, (1.20) ** (1 / 365.25) - 1])

    def test_stale_risk_free_quotes_fail(self):
        yields = pd.Series([5.], index=pd.to_datetime(["2024-01-01"]))
        with self.assertRaisesRegex(DataError, "stale"):
            risk_free_returns(pd.to_datetime(["2024-01-20", "2024-01-21"]), yields_percent=yields)

    def test_fred_requires_explicit_key(self):
        with TemporaryDirectory() as directory:
            with self.assertRaisesRegex(DataError, "requires"):
                fetch_fred_yields(self.config, "", Path(directory))

    def test_demo_is_deterministic_and_labelled(self):
        first, second = generate_demo(self.config), generate_demo(self.config)
        pd.testing.assert_frame_equal(first.prices, second.prices)
        self.assertTrue(first.metadata["synthetic"])

    def test_invalid_configuration_fails_early(self):
        with self.assertRaises(ValueError):
            Config(weights={"A": .9})
        with self.assertRaises(ValueError):
            Config(cost_bps=-1)
        with self.assertRaises(ValueError):
            Config(start="2025-01-01", end="2024-01-01")

    def test_pipeline_uses_one_common_calendar(self):
        analysis = run_analysis(self.config, mode="demo")
        for result in analysis.results.values():
            self.assertTrue(result.nav.index.equals(analysis.prices.index))
            self.assertTrue(result.returns.index.equals(analysis.risk_free.index))
        self.assertTrue(analysis.metadata["data"]["synthetic"])


if __name__ == "__main__":
    unittest.main()


In [ ]:
# @title Write documentation and project metadata
supporting_files = {'README.md': '# Portfolio Lab — Multi-Asset Performance & Risk\n'
              '\n'
              'A complete Colab and Python research project for comparing a rebalanced multi-asset '
              'portfolio, buy-and-hold, and a configurable benchmark.\n'
              '\n'
              '![Synthetic demonstration](examples/executive_summary.png)\n'
              '\n'
              '**The included example uses synthetic data. Its results are not historical ETF performance.** '
              'Switch to Yahoo mode to download real adjusted prices.\n'
              '\n'
              '## Start in Google Colab\n'
              '\n'
              '1. Open https://colab.research.google.com/.\n'
              '2. Choose **File → Upload notebook** and upload '
              '**notebooks/Multi_Asset_Portfolio_Dashboard.ipynb**.\n'
              '3. Leave **DATA_MODE = "demo"** for the reproducible offline example.\n'
              '4. Select **Runtime → Run all**.\n'
              '5. Download the generated dashboard and result bundle from the final cell.\n'
              '\n'
              'The notebook is self-contained: its visible source cells create the Python package. You do '
              'not need to upload the repository as well. To use actual prices, change DATA_MODE to "yahoo" '
              'and run all cells again. Yahoo mode installs the optional yfinance adapter. To use a CSV, '
              'upload it to Colab and set CSV_PATH to its absolute path.\n'
              '\n'
              '## Run locally\n'
              '\n'
              'Python 3.10 or newer is required; Python 3.12 was used for local verification.\n'
              '\n'
              '~~~bash\n'
              'python -m venv .venv\n'
              'source .venv/bin/activate\n'
              'python -m pip install -r requirements.txt\n'
              'python -m portfolio_lab --mode demo\n'
              'python -m unittest discover -s tests -v\n'
              '~~~\n'
              '\n'
              'On Windows, activate with **.venv\\Scripts\\activate**. Open **reports/dashboard.html** in a '
              'browser after the run.\n'
              '\n'
              '~~~bash\n'
              '# Real adjusted prices: network access and Yahoo availability required.\n'
              'python -m pip install -r requirements-live.txt\n'
              'python -m portfolio_lab --mode yahoo --start 2015-01-01 --end 2026-01-01\n'
              '\n'
              '# Read a user-supplied adjusted-price file.\n'
              'python -m portfolio_lab --mode csv --csv path/to/adjusted_prices.csv\n'
              '\n'
              '# Change portfolio mechanics.\n'
              'python -m portfolio_lab --mode demo --rebalance quarterly --cost-bps 10\n'
              '~~~\n'
              '\n'
              'For historical risk-free yield proxies, set FRED_API_KEY in your environment and pass '
              '**--fred**. In Colab, store the key in **Secrets → FRED_API_KEY**, enable notebook access, '
              'and set USE_FRED = True. Credentials are not saved in reports.\n'
              '\n'
              '## What is implemented\n'
              '\n'
              '- Eight configurable ETF allocations; a default 60% SPY / 40% AGG benchmark.\n'
              '- Holdings drift between rebalances, with close trades affecting subsequent returns.\n'
              '- Transaction costs funded from NAV using an exact self-financing cost equation.\n'
              '- Daily return and dollar P&L reconciliation, benchmark-relative metrics, and risk '
              'attribution.\n'
              '- Strict missing-data handling, explicit common-history trimming, bounded API retries and '
              'checksummed snapshots.\n'
              '- Ten professional PNG/SVG chart exports and a self-contained interactive HTML dashboard.\n'
              '- CSV outputs, a configuration/provenance manifest, financial correctness tests, and GitHub '
              'CI.\n'
              '\n'
              '**[Read the complete 15-part project guide](docs/PROJECT_GUIDE.md)** for architecture, '
              'implementation steps, formulas, API choices, outputs, a résumé description, and extensions.\n'
              '\n'
              '## Portfolio conventions\n'
              '\n'
              'The first date represents an already-invested portfolio. Initial purchase and final '
              'liquidation costs are excluded consistently. Positions are fractional total-return units, not '
              'executable broker share counts. All assets are USD-listed; no leverage, external deposits, '
              'taxes, or separate cash allocation are modeled.\n'
              '\n'
              'Rebalancing occurs at the close of the first observed session in a new calendar period. A '
              'cost of 5 basis points means 0.0005 times the gross dollar amount bought plus sold. The '
              'default cash-rate assumption is a clearly labelled fixed 2% effective annual rate, not a '
              'reconstructed historical rate.\n'
              '\n'
              "Adjusted-close returns already incorporate the provider's distribution and split adjustments. "
              'The engine does not add dividends a second time. Live data may be revised by the provider; '
              'preserve the run manifest and cached snapshot for reproducibility.\n'
              '\n'
              '## Data rights and sources\n'
              '\n'
              '- [yfinance API '
              'documentation](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)\n'
              '- [yfinance project and Yahoo usage notes](https://github.com/ranaroussi/yfinance)\n'
              '- [FRED observations '
              'API](https://fred.stlouisfed.org/docs/api/fred/series_observations.html)\n'
              '- [FRED DGS3MO series definition](https://fred.stlouisfed.org/series/DGS3MO)\n'
              '\n'
              "yfinance is unofficial. Check the provider's data-use terms before redistributing downloaded "
              'market data. Generated caches and reports are ignored by git; the included examples contain '
              'only synthetic data.\n'
              '\n'
              '## Verification\n'
              '\n'
              'See **docs/VALIDATION.md** for actual execution results and the boundaries of live-provider '
              'and browser testing. Undefined statistical ratios remain NaN/N/A.\n'
              '\n'
              '## Publishing on GitHub\n'
              '\n'
              'Upload this project directory to a repository you own. Keep the example image and clearly '
              'labelled synthetic outputs for an immediately reviewable demonstration. Run Yahoo mode '
              'yourself, preserve its manifest, and write a short research memo interpreting actual results '
              'before presenting empirical findings in an interview.\n'
              '\n'
              'The original repository distribution includes tools/build_notebook.py to regenerate the '
              'notebook. The notebook embeds source code rather than downloading it from an unpinned '
              'repository.\n',
 'requirements.txt': 'numpy>=1.26,<3\npandas>=2.2,<3\nmatplotlib>=3.8,<4\n',
 'requirements-live.txt': '-r requirements.txt\nyfinance>=0.2.54,<2\n',
 'pyproject.toml': '[build-system]\n'
                   'requires = ["setuptools>=68"]\n'
                   'build-backend = "setuptools.build_meta"\n'
                   '\n'
                   '[project]\n'
                   'name = "portfolio-lab-dashboard"\n'
                   'version = "1.0.0"\n'
                   'description = "Auditable multi-asset portfolio performance and risk research"\n'
                   'requires-python = ">=3.10"\n'
                   'dependencies = ["numpy>=1.26,<3", "pandas>=2.2,<3", "matplotlib>=3.8,<4"]\n'
                   '\n'
                   '[project.optional-dependencies]\n'
                   'live = ["yfinance>=0.2.54,<2"]\n'
                   '\n'
                   '[project.scripts]\n'
                   'portfolio-lab = "portfolio_lab.__main__:main"\n'
                   '\n'
                   '[tool.setuptools.packages.find]\n'
                   'include = ["portfolio_lab*"]\n',
 '.gitignore': '__pycache__/\n'
               '*.py[cod]\n'
               '.venv/\n'
               '.ipynb_checkpoints/\n'
               '.env\n'
               'data/cache/*\n'
               '!data/cache/.gitkeep\n'
               'reports/*\n'
               '!reports/.gitkeep\n'
               '*.egg-info/\n'
               'build/\n'
               'dist/\n',
 '.github/workflows/tests.yml': 'name: Financial correctness\n'
                                'on: [push, pull_request]\n'
                                'jobs:\n'
                                '  test:\n'
                                '    runs-on: ubuntu-latest\n'
                                '    steps:\n'
                                '      - uses: actions/checkout@v4\n'
                                '      - uses: actions/setup-python@v5\n'
                                '        with:\n'
                                '          python-version: "3.12"\n'
                                '      - run: python -m pip install -r requirements.txt\n'
                                '      - run: python -m unittest discover -s tests -v\n'
                                '      - run: python -m portfolio_lab --mode demo --start 2023-01-01 --end '
                                '2025-01-01\n'
                                '        env:\n'
                                '          MPLBACKEND: Agg\n',
 'docs/PROJECT_GUIDE.md': '# Multi-Asset Portfolio Performance and Risk Dashboard\n'
                          '\n'
                          'This guide covers the complete project specification and implemented research '
                          'workflow. The source files and Colab notebook contain the full implementation.\n'
                          '\n'
                          '## 1. Project overview\n'
                          '\n'
                          'Build a reproducible research system that answers:\n'
                          '\n'
                          '- How did a multi-asset allocation perform?\n'
                          '- Which assets produced its profits and losses?\n'
                          '- How much risk came from each asset?\n'
                          '- Did rebalancing improve outcomes after trading costs?\n'
                          '- How did the portfolio compare with a 60/40 benchmark?\n'
                          '- How would current holdings respond to specified shocks?\n'
                          '\n'
                          'The implemented default universe is SPY, VEA, VWO, AGG, IEF, TIP, GLD and VNQ. '
                          'Target weights are 30%, 15%, 5%, 20%, 10%, 5%, 10% and 5%, respectively. The '
                          'example allocation is a research configuration, not an optimized recommendation.\n'
                          '\n'
                          'There are three explicit data modes: deterministic synthetic demonstration, Yahoo '
                          'adjusted-price download, and user-supplied CSV. API failure never silently '
                          'switches a historical run to simulated data.\n'
                          '\n'
                          'Scope: daily observations; long-only fully invested portfolios; one USD reporting '
                          'currency; no external cash flows; fractional total-return units.\n'
                          '\n'
                          '## 2. Real-world finance use case\n'
                          '\n'
                          'An asset-management analyst receives a proposed allocation and needs an auditable '
                          'portfolio review for an investment meeting. The system produces performance, '
                          'risk, allocation and cost analysis using a consistent set of assumptions.\n'
                          '\n'
                          'Applications include investment reporting, wealth-management allocation reviews, '
                          'benchmark comparisons, treasury investment analysis and interview research '
                          'demonstrations.\n'
                          '\n'
                          'The project emphasizes return accounting and interpretation. An attractive '
                          'performance chart alone cannot establish that a portfolio is suitable, investable '
                          'or likely to outperform.\n'
                          '\n'
                          '## 3. System architecture\n'
                          '\n'
                          '| Layer | Implementation | Responsibility |\n'
                          '|---|---|---|\n'
                          '| Configuration | portfolio_lab/config.py | Dates, weights, capital, costs, '
                          'frequency and validation |\n'
                          '| Data adapters | portfolio_lab/data.py | Demo, Yahoo, CSV, FRED, caching and '
                          'provenance |\n'
                          '| Validation | clean_prices | Common history, numeric checks, duplicate/gap '
                          'checks |\n'
                          '| Simulation | portfolio_lab/engine.py | Holdings, close rebalances, cost '
                          'funding, ledgers |\n'
                          '| Analytics | portfolio_lab/analytics.py | Metrics, drawdowns, contribution and '
                          'scenario analysis |\n'
                          '| Visualization | portfolio_lab/charts.py | Ten consistent Matplotlib charts |\n'
                          '| Dashboard | portfolio_lab/report.py | Offline HTML, embedded SVG, native '
                          'browser interaction |\n'
                          '| Orchestration | portfolio_lab/pipeline.py | Run configuration and reproducible '
                          'export bundle |\n'
                          '| Entry points | notebook / __main__.py | Colab teaching workflow and local '
                          'command line |\n'
                          '| Verification | tests/ | Hand-calculated financial cases and provider-contract '
                          'fixtures |\n'
                          '\n'
                          '~~~mermaid\n'
                          'flowchart TD\n'
                          '    A["Configuration"] --> B["Data adapters and cache"]\n'
                          '    B --> C["Price validation"]\n'
                          '    C --> D["Holdings engine"]\n'
                          '    D --> E["Risk and performance analytics"]\n'
                          '    E --> F["Charts and HTML dashboard"]\n'
                          '    E --> G["CSV outputs and manifest"]\n'
                          '    H["Financial correctness tests"] -.-> C\n'
                          '    H -.-> D\n'
                          '    H -.-> E\n'
                          '~~~\n'
                          '\n'
                          'Calculation functions are separate from API calls and rendering. This makes the '
                          'accounting reusable and allows tests to run without network access.\n'
                          '\n'
                          '## 4. Required APIs and data sources\n'
                          '\n'
                          '| Source | Required? | Authentication | Fields used |\n'
                          '|---|---|---|---|\n'
                          '| Synthetic generator | Default demonstration | None | Reproducible positive '
                          'total-return indices |\n'
                          '| Yahoo Finance via yfinance | For live-price mode | No API key in this adapter | '
                          'Daily adjusted close; adjustment option set explicitly |\n'
                          '| FRED DGS3MO | Optional risk-free proxy | FRED API key | Date and annualized '
                          '3-month Treasury yield |\n'
                          '| User CSV | Optional alternative | None | Date and one adjusted-price column per '
                          'ticker |\n'
                          '\n'
                          'The CSV must use a Date header, ISO dates, and finite positive USD adjusted-price '
                          'columns matching all configured portfolio and benchmark tickers. A sample is '
                          'included in examples/synthetic_adjusted_prices.csv. Its values are synthetic.\n'
                          '\n'
                          'SEC EDGAR is not needed for a price-based portfolio dashboard. Alpha Vantage, '
                          'Financial Modeling Prep and Massive can be added as alternate adapters, but are '
                          'not required dependencies.\n'
                          '\n'
                          'Official source references:\n'
                          '\n'
                          '1. [yfinance download '
                          'parameters](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html): '
                          'explicitly control adjustments and understand its column schema.\n'
                          '2. [yfinance usage notes](https://github.com/ranaroussi/yfinance): unofficial '
                          'package and provider data-use considerations.\n'
                          '3. [FRED observations '
                          'API](https://fred.stlouisfed.org/docs/api/fred/series_observations.html): request '
                          'parameters and observation format.\n'
                          '4. [DGS3MO definition](https://fred.stlouisfed.org/series/DGS3MO): yields quoted '
                          'in percent per annum on an investment basis.\n'
                          '\n'
                          'Historical adjusted prices are not a point-in-time security master. This fixed '
                          'ETF universe does not remove selection or survivorship bias.\n'
                          '\n'
                          '## 5. Required Python libraries\n'
                          '\n'
                          '| Library | Purpose | Required in offline mode? |\n'
                          '|---|---|---|\n'
                          '| numpy | Vectorized returns, holdings arrays and covariance algebra | Yes |\n'
                          '| pandas | Time-series alignment, period aggregation and tables | Yes |\n'
                          '| matplotlib | Professional static and embedded SVG charts | Yes |\n'
                          '| yfinance | Yahoo historical-price adapter | Only for Yahoo mode |\n'
                          '| Python standard library | Configuration, HTTP, caching, JSON, CLI, tests and '
                          'ZIP export | Included |\n'
                          '| IPython.display | Optional inline display in Colab/Jupyter | Bundled with Colab '
                          '|\n'
                          '\n'
                          'The HTML report uses small native JavaScript controls and embedded SVG. It does '
                          'not need a chart CDN, a server, Plotly, Streamlit or a running Python session '
                          'after export.\n'
                          '\n'
                          '## 6. Folder/file structure\n'
                          '\n'
                          '| Path | Purpose |\n'
                          '|---|---|\n'
                          '| README.md | Entry point, local setup, Colab instructions and assumptions |\n'
                          '| requirements.txt | Small numerical and plotting dependency set |\n'
                          '| requirements-live.txt | Optional Yahoo dependency |\n'
                          '| pyproject.toml | Installable package metadata and console entry point |\n'
                          '| notebooks/Multi_Asset_Portfolio_Dashboard.ipynb | Complete self-contained Colab '
                          'workflow |\n'
                          '| portfolio_lab/config.py | Validated portfolio settings |\n'
                          '| portfolio_lab/data.py | Providers, cache, cleaning and risk-free alignment |\n'
                          '| portfolio_lab/engine.py | Holdings-based portfolio simulation |\n'
                          '| portfolio_lab/analytics.py | Performance, tail risk, drawdown and attribution '
                          '|\n'
                          '| portfolio_lab/charts.py | Consistent chart factory and PNG/SVG exports |\n'
                          '| portfolio_lab/report.py | Standalone interactive HTML |\n'
                          '| portfolio_lab/pipeline.py | Analysis orchestration and output manifest |\n'
                          '| portfolio_lab/__main__.py | Local command-line entry point |\n'
                          '| tests/test_finance.py | Financial identities and hand-calculated examples |\n'
                          '| tests/test_data.py | Data quality, API schemas, caching and timing |\n'
                          '| tools/build_notebook.py | Regenerate the notebook from the source files |\n'
                          '| docs/PROJECT_GUIDE.md | This complete project design |\n'
                          '| docs/VALIDATION.md | Executed checks and verification limitations |\n'
                          '| examples/ | Labelled synthetic data and example charts |\n'
                          '| data/cache/ | Checksummed provider snapshots, ignored by git |\n'
                          '| reports/ | Generated dashboard, tables, manifest and figures |\n'
                          '| .github/workflows/tests.yml | Automatic correctness checks after publishing |\n'
                          '\n'
                          'Colab creates the same reusable package structure in its runtime. It is '
                          'intentionally possible to move the project into a conventional repository without '
                          'rewriting the financial logic.\n'
                          '\n'
                          '## 7. Step-by-step build guide\n'
                          '\n'
                          '1. **Select the scope.** Fix the reporting currency, date range, invested '
                          'starting value, target weights, benchmark and trading-cost convention.\n'
                          '2. **Validate configuration.** Reject negative or non-finite weights and require '
                          'weights to sum to one.\n'
                          '3. **Collect data.** Choose one explicit source mode; retrieve all portfolio and '
                          'benchmark symbols together.\n'
                          '4. **Audit observations.** Reject duplicate dates, nonnumeric/nonpositive prices '
                          'and interior gaps. Trim to common history and report the trimming.\n'
                          '5. **Build return series.** Divide consecutive adjusted prices; never substitute '
                          'missing observations with zero returns.\n'
                          "6. **Run holdings accounting.** Apply each day's price changes to previous "
                          'holdings, then rebalance at the scheduled close.\n'
                          '7. **Fund costs.** Solve for investable wealth after costs instead of allocating '
                          'money that has already been spent.\n'
                          '8. **Compare strategies.** Run configured rebalancing, buy-and-hold and the '
                          'monthly benchmark on identical dates.\n'
                          '9. **Compute risk-free proxies.** Use the explicit constant assumption or '
                          'strictly backward-aligned FRED yields.\n'
                          '10. **Calculate analytics.** Produce growth, risk, tail losses, benchmark '
                          'metrics, rolling series and contributions.\n'
                          '11. **Verify identities.** Asset contributions plus costs must equal daily '
                          'portfolio returns; dollar contributions must equal total P&L.\n'
                          '12. **Render outputs.** Save the HTML dashboard, professional figures, numeric '
                          'CSVs and provenance manifest.\n'
                          '13. **Write the research interpretation.** Explain what drove results and how '
                          'conclusions change with costs, dates, weights and benchmarks.\n'
                          '\n'
                          'Estimated learning-and-build effort: roughly 20–35 focused hours with basic '
                          'Python and finance knowledge. The supplied implementation is complete; '
                          'understanding and extending it is the substantive learning work.\n'
                          '\n'
                          '## 8. Data collection pipeline\n'
                          '\n'
                          'Yahoo downloads use auto_adjust=False and select Adj Close explicitly. The parser '
                          'handles flat single-ticker output and both common orientations of MultiIndex '
                          'columns. Raw Close is never silently substituted.\n'
                          '\n'
                          'Queries are keyed by provider, fields, symbols and date interval. Completed data '
                          'and metadata files are written atomically. Metadata includes collection time, '
                          'query, adjustment convention and SHA-256. The cache is a frozen snapshot until an '
                          'explicit refresh is requested.\n'
                          '\n'
                          'Downloads have bounded retries. A failed live request returns an actionable '
                          'error. It does not claim stale or generated data are current observations.\n'
                          '\n'
                          'FRED requests use the observations API and a 30-day initial lookback. Credentials '
                          'stay in environment variables or Colab Secrets. API keys are excluded from logs, '
                          'cache metadata and manifests.\n'
                          '\n'
                          'The manifest records the actual data interval, requested configuration, '
                          'data-source flags, validation findings, software versions, and checksums for '
                          'exported CSVs.\n'
                          '\n'
                          '## 9. Data cleaning and feature engineering\n'
                          '\n'
                          '**Dates:** parse, normalize daily labels, remove timezone information while '
                          'preserving exchange-local dates, reject duplicates, sort, and apply the '
                          'configured start-inclusive/end-exclusive interval.\n'
                          '\n'
                          '**Prices:** require finite, positive observations; select all portfolio and '
                          'benchmark symbols; trim to the first and last dates where the full universe '
                          'exists. Interior missing cells are errors. Gaps longer than seven calendar days '
                          'require investigation.\n'
                          '\n'
                          '**Diagnostics:** flag unusually large daily moves over 35%, trimmed history and '
                          'stale trailing coverage. These checks do not constitute a complete '
                          'exchange-calendar or corporate-action audit.\n'
                          '\n'
                          '**Features:** simple asset returns, portfolio net returns, excess returns, '
                          'drawdowns, monthly compounded returns, rolling volatility, rolling Sharpe, '
                          'rolling beta, gross turnover, P&L and risk contributions.\n'
                          '\n'
                          'The model uses adjusted total-return units. It does not combine adjusted returns '
                          'with separately added cash dividends.\n'
                          '\n'
                          'For the FRED proxy, a yield observation must strictly predate the start of the '
                          'return interval. Forward-carrying a known rate is allowed for a maximum of 10 '
                          'calendar days; prices are never carried forward. The latest FRED vintage is used, '
                          'not a historical ALFRED reconstruction.\n'
                          '\n'
                          '## 10. Core models and algorithms\n'
                          '\n'
                          '### Asset and portfolio returns\n'
                          '\n'
                          'For adjusted price P, asset return is r(i,t) = P(i,t) / P(i,t−1) − 1.\n'
                          '\n'
                          'If H(i,t−1) is the prior closing dollar holding, pre-trade value is:\n'
                          '\n'
                          'H_before(i,t) = H(i,t−1) × [1 + r(i,t)].\n'
                          '\n'
                          "Today's market P&L is H(i,t−1) × r(i,t). Rebalancing at today's close cannot "
                          'change that already-earned P&L.\n'
                          '\n'
                          '### Transaction-cost equation\n'
                          '\n'
                          'For pre-trade values H_before, target weights w and per-dollar cost c, solve:\n'
                          '\n'
                          'V_after + c × Σ |w(i) × V_after − H_before(i)| = V_before.\n'
                          '\n'
                          'The implementation uses a bounded 60-step bisection. Post-trade values are w × '
                          'V_after. This pays the cost from portfolio value and preserves the accounting '
                          'identity.\n'
                          '\n'
                          'Gross turnover = gross bought-plus-sold notional / pre-trade NAV. A convention '
                          'that halves turnover would produce a different statistic; this project explicitly '
                          'uses gross turnover.\n'
                          '\n'
                          '### Rebalancing\n'
                          '\n'
                          'Calendar rules are monthly, quarterly, annual or none. Trading happens at the '
                          "first observed session's close in a new period. Target weights apply to the "
                          'following return interval. The benchmark always rebalances monthly, which is '
                          'recorded explicitly.\n'
                          '\n'
                          '### Attribution\n'
                          '\n'
                          'Daily return contribution from asset i is asset P&L(i,t) / NAV(t−1). Cost '
                          "contribution is −fees(t) / NAV(t−1). Their sum equals the portfolio's net daily "
                          'return.\n'
                          '\n'
                          "Dollar P&L attribution sums each asset's daily dollar P&L and adds trading costs "
                          'as a negative component. Contributions sum exactly to ending NAV minus starting '
                          'capital. Summing daily return contributions would not generally equal a '
                          'compounded total return.\n'
                          '\n'
                          '### Risk contribution\n'
                          '\n'
                          'Given current weights w and annualized covariance Σ:\n'
                          '\n'
                          '- Portfolio volatility: σ = sqrt(wᵀΣw).\n'
                          '- Component volatility: RC(i) = w(i) × (Σw)(i) / σ.\n'
                          '- Share of volatility: RC(i) / σ.\n'
                          '\n'
                          'The components sum to total volatility. Negative contributions may represent '
                          'hedging and are retained. Covariance uses the full observed sample for '
                          'descriptive analysis; it is never fed into earlier backtest decisions.\n'
                          '\n'
                          '### Stress scenarios\n'
                          '\n'
                          'For hypothetical simple asset shocks s, portfolio shock is wᵀs. Scenario '
                          'definitions are exported. They are instantaneous assumptions without assigned '
                          'probabilities. Custom tickers disable the default scenario panel until explicit '
                          'shocks are supplied.\n'
                          '\n'
                          '### Complexity and optimization\n'
                          '\n'
                          'Return arrays are computed once. The holdings loop is O(T×N), which is '
                          'appropriate because holdings are path-dependent. Cost solving occurs only on '
                          'rebalance dates; each solve is O(60×N). The default eight-asset daily dataset '
                          'comfortably fits a standard Colab CPU runtime.\n'
                          '\n'
                          '## 11. Visualizations and dashboard components\n'
                          '\n'
                          'The standalone HTML dashboard has four tabs:\n'
                          '\n'
                          '1. **Overview:** selected-strategy KPI cards, interactive NAV chart, benchmark '
                          'comparison, drawdowns and monthly heatmap.\n'
                          '2. **Risk:** rolling volatility, asset correlation, capital-versus-risk '
                          'contribution, and hypothetical stress outcomes.\n'
                          '3. **Attribution:** allocation history and reconciled dollar P&L.\n'
                          '4. **Data & assumptions:** source provenance, cash-rate assumption, cost '
                          'conventions and data-quality findings.\n'
                          '\n'
                          'The strategy selector updates KPI cards and highlights a curve. The 1Y/3Y/All '
                          'buttons change the plotted range only; KPI cards explicitly remain full-sample '
                          'statistics. Hover displays exact values. Detailed asset panels describe the '
                          'configured Portfolio.\n'
                          '\n'
                          'Ten figures are exported in PNG and SVG: growth, drawdowns, rolling volatility, '
                          'monthly returns, correlation, weights, P&L attribution, risk contributions, '
                          'stress, and a composite executive summary.\n'
                          '\n'
                          '## 12. Performance metrics\n'
                          '\n'
                          'Let A = 252, e = portfolio return minus matching cash-return proxy, b = benchmark '
                          'excess return, and a = portfolio return minus benchmark return. Standard '
                          'deviations use sample ddof=1.\n'
                          '\n'
                          '| Metric | Convention |\n'
                          '|---|---|\n'
                          '| Total return | Ending NAV / starting NAV − 1 |\n'
                          '| CAGR | (Ending NAV / starting NAV)^(365.25 / elapsed calendar days) − 1 |\n'
                          '| Annualized volatility | Daily return standard deviation × sqrt(A) |\n'
                          '| Sharpe ratio | Mean(e) / standard deviation(e) × sqrt(A) |\n'
                          '| Sortino ratio | Mean(e) × sqrt(A) / sqrt(mean(min(e,0)^2)); includes all days '
                          'in the denominator average |\n'
                          '| Drawdown | NAV / running maximum NAV − 1 |\n'
                          '| Maximum drawdown | Most negative drawdown |\n'
                          '| Calmar ratio | CAGR / absolute maximum drawdown |\n'
                          '| Tracking error | Standard deviation(a) × sqrt(A) |\n'
                          '| Information ratio | Mean(a) / standard deviation(a) × sqrt(A) |\n'
                          '| Beta | Covariance(e,b) / variance(b) |\n'
                          '| Arithmetic annual alpha | [Mean(e) − beta × Mean(b)] × A |\n'
                          '| Historical daily VaR | Configured quantile of daily loss L = −return |\n'
                          '| Historical daily ES | Mean observed loss at or above the VaR threshold |\n'
                          '| Positive-day fraction | Fraction of portfolio net returns greater than zero |\n'
                          '| Longest drawdown | Peak-to-recovery calendar days, or peak-to-end for an '
                          'ongoing episode |\n'
                          '| Annual gross turnover | Sum of gross turnover / elapsed calendar years |\n'
                          '| Fees | Sum of dollar rebalancing costs |\n'
                          '\n'
                          'VaR and ES use an empirical daily loss convention: positive values mean losses; '
                          'negative values can occur if even the relevant tail consists of gains. These are '
                          'descriptive tail estimates, not a bank regulatory-capital implementation. '
                          'Undefined ratios remain NaN/N/A.\n'
                          '\n'
                          'The fixed risk-free proxy compounds by actual calendar days using actual/365.25. '
                          'FRED DGS3MO is an annualized quoted yield; treating it as an effective annual '
                          'cash rate is a documented approximation, not an observed Treasury investment '
                          'return. No standalone cash sleeve earns this proxy in portfolio NAV.\n'
                          '\n'
                          '## 13. Final deliverables\n'
                          '\n'
                          '- A standalone, cell-labelled Colab notebook with full visible module source.\n'
                          '- A reusable installable Python package and CLI.\n'
                          '- This 15-part guide and setup README.\n'
                          '- A financial correctness and data-contract test suite plus CI workflow.\n'
                          '- A labelled synthetic input example and executive-summary chart.\n'
                          '- An interactive HTML dashboard that works offline.\n'
                          '- Daily ledgers, positions, weights, returns, monthly tables, rolling metrics, '
                          'drawdown episodes, contributions and stress CSVs.\n'
                          '- Ten PNG/SVG charts and a manifest containing assumptions, software versions and '
                          'data checksums.\n'
                          '\n'
                          'Interpretation template for a research memo: **question → data and dates → '
                          'assumptions → performance and risk findings → attribution → sensitivity analysis '
                          '→ limitations → next experiment**. Report observations actually measured in your '
                          'run.\n'
                          '\n'
                          '## 14. Résumé description\n'
                          '\n'
                          '**Project title:** Multi-Asset Portfolio Performance & Risk Analytics — Python\n'
                          '\n'
                          '**Suggested bullet:**\n'
                          '\n'
                          'Built a reproducible eight-asset portfolio analytics system in Python, '
                          'implementing holdings-based rebalancing, transaction-cost accounting, benchmark '
                          'comparisons, and performance/risk attribution; delivered a self-contained '
                          'dashboard, documented data pipelines, and automated financial correctness tests.\n'
                          '\n'
                          '**Technical keywords:** Python, NumPy, pandas, Matplotlib, time-series analysis, '
                          'portfolio accounting, risk analytics, API integration, testing, Google Colab.\n'
                          '\n'
                          '**Interview talking points:**\n'
                          '\n'
                          '- Why do buy-and-hold weights drift?\n'
                          '- How can an apparently small cost-model shortcut violate self-financing?\n'
                          '- Why do adjusted prices and separately added dividends double-count '
                          'distributions?\n'
                          '- Why do missing-price fills alter measured volatility and correlation?\n'
                          '- Why is the arithmetic sum of daily return contributions different from a '
                          'compounded total return?\n'
                          '- How does the selected benchmark change beta and active-risk interpretation?\n'
                          '\n'
                          'Quantify returns or speed improvements in a résumé only after measuring and '
                          'documenting them yourself.\n'
                          '\n'
                          '## 15. Potential upgrades\n'
                          '\n'
                          '| Upgrade | Finance benefit | Engineering work |\n'
                          '|---|---|---|\n'
                          '| External cash flows, TWR and money-weighted IRR | Support investor account '
                          'reporting | Cash-flow ledger and unitization |\n'
                          '| Explicit cash sleeve | Model uninvested balances and financing | Cash accrual '
                          'and portfolio cash constraints |\n'
                          '| FX translation and currency attribution | Support portfolios with different '
                          'listing currencies | FX calendar alignment and local/base return decomposition |\n'
                          '| Historical security master | Reduce selection/survivorship problems | '
                          'Membership, identifier changes and delisting records |\n'
                          '| Exchange-calendar validation | Detect missing sessions across the whole '
                          'universe | Integrate exchange calendars and holiday logic |\n'
                          '| ALFRED vintages | Reconstruct historical economic information | Vintage-aware '
                          'observation storage |\n'
                          '| Factor attribution | Separate systematic exposure from residual performance | '
                          'Factor data and robust regression inference |\n'
                          '| Covariance shrinkage and risk budgeting | Explore more stable allocation '
                          'methods | Rolling estimation and constrained optimization |\n'
                          '| Spread/impact models | Improve trading-cost realism | Liquidity data and '
                          'nonlinear execution assumptions |\n'
                          '| Bootstrap confidence intervals | Quantify estimate uncertainty | Block '
                          'resampling and statistical reporting |\n'
                          '| Broker reconciliation | Compare research output with account records | '
                          'Trade/cash/dividend ledger and reconciliation |\n'
                          '| Scheduled deployment | Generate recurring reports | Durable compute, '
                          'monitoring, secrets and provider entitlements |\n'
                          '\n'
                          'Start with cost, allocation and date-range sensitivity analysis before adding '
                          'more complex predictive models.\n',
 'docs/VALIDATION.md': '# Verification record\n'
                       '\n'
                       '## Executed checks\n'
                       '\n'
                       "- **34 automated tests passed** using Python's standard-library unittest runner.\n"
                       '- **All 21 Colab code cells executed successfully** in order in a clean local '
                       "directory. The execution harness implements the notebook's standard %%writefile "
                       'cells and executes the remaining Python in a shared namespace.\n'
                       "- The notebook's embedded financial/data test suite passed.\n"
                       '- End-to-end synthetic analysis produced the HTML dashboard, CSV tables, manifest, '
                       'chart exports, sensitivity experiment and downloadable result bundle.\n'
                       '- Source files compiled successfully.\n'
                       '- Generated charts were visually inspected for legibility, clipping, labels and '
                       'source identification.\n'
                       '- Dashboard JavaScript was checked with a dependency-free DOM-contract harness: '
                       'initial render, strategy KPI selection, chart ranges, tab switching, hover tooltips '
                       'and narrow-width SVG coordinates.\n'
                       '\n'
                       '## Financial cases covered\n'
                       '\n'
                       '1. Buy-and-hold results match independent fixed-unit arithmetic.\n'
                       '2. Weights drift after relative price movements.\n'
                       '3. A close rebalance cannot change the return earned before that close.\n'
                       '4. Cost funding matches hand-derived two-asset and full-rotation solutions.\n'
                       '5. Positions add to NAV and weights add to one.\n'
                       '6. Dollar P&L and daily return contributions reconcile.\n'
                       '7. Changes to future prices do not change past NAV.\n'
                       '8. Constant prices produce no economically meaningful turnover or costs.\n'
                       '9. Invalid long-only weights are rejected.\n'
                       '10. Drawdown depth, recovery and ongoing durations are handled.\n'
                       '11. Monthly returns compound to the whole-period result.\n'
                       '12. An identical benchmark gives beta one and zero active tracking error.\n'
                       '13. Undefined ratios remain NaN rather than becoming infinite.\n'
                       '14. Historical tail metrics use the correct loss sign.\n'
                       '15. CAGR uses elapsed calendar time.\n'
                       '16. Sortino uses an unconditional lower-partial-moment denominator.\n'
                       '17. Euler component risks sum to total volatility.\n'
                       '18. Risk-free proxies accrue over actual calendar intervals.\n'
                       '19. A FRED quote must predate the start of the return interval.\n'
                       '20. Duplicate dates, nonpositive prices, infinite values and interior gaps fail '
                       'explicitly.\n'
                       '21. Yahoo adjusted-close schema variants are parsed without silently using raw '
                       'close.\n'
                       '22. Caches are reused and source failures do not become synthetic data.\n'
                       '\n'
                       '## Environment\n'
                       '\n'
                       'The calculation core was exercised with Python 3.12, NumPy 2.3.5, pandas 2.2.3 and '
                       "Matplotlib 3.10.8. A run's manifest records the versions actually used in that "
                       'environment.\n'
                       '\n'
                       '## Verification boundaries\n'
                       '\n'
                       '- Live Yahoo and FRED calls were **not executed end to end** in the build '
                       'environment. Optional package installation was unavailable, and no user API key was '
                       'supplied. Yahoo adapter schemas and error behavior were tested with deterministic '
                       'fixtures.\n'
                       '- The notebook was computationally executed with a local cell harness, **not inside '
                       'an authenticated Google Colab session**. Colab Secrets, its download UI and notebook '
                       'iframe rendering need a user-side Colab run.\n'
                       '- A Chromium executable was unavailable, so a real browser layout test could not be '
                       'completed. JavaScript behavior was checked with the DOM-contract harness; static PNG '
                       'charts were visually inspected.\n'
                       '- Synthetic paths establish software behavior, not an empirical investment result.\n'
                       '- The included tests do not establish production trading suitability or exhaustive '
                       'data-provider compatibility.\n'
                       '\n'
                       '## Repeat the checks\n'
                       '\n'
                       '~~~bash\n'
                       'python -m unittest discover -s tests -v\n'
                       'python -m portfolio_lab --mode demo\n'
                       'python tools/build_notebook.py\n'
                       'python tools/check_notebook.py\n'
                       'node tools/check_dashboard.cjs\n'
                       '~~~\n'
                       '\n'
                       'Node is only needed for the optional JavaScript smoke test. Neither the Python '
                       'application nor the exported HTML needs a Node server.\n'}
for relative, content in supporting_files.items():
    destination = PROJECT_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding='utf-8')
print('Documentation and project metadata written.')


In [ ]:
# @title Run the financial correctness suite
import importlib
import unittest

importlib.invalidate_caches()
for module in list(sys.modules):
    if module == "portfolio_lab" or module.startswith("portfolio_lab.") or module in {"test_finance", "test_data"}:
        del sys.modules[module]
suite = unittest.defaultTestLoader.discover(str(PROJECT_ROOT / "tests"))
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
if not test_result.wasSuccessful():
    raise RuntimeError("Financial correctness checks failed. Resolve these before interpreting outputs.")


In [ ]:
# @title Collect data and run the portfolio comparison
from portfolio_lab.config import Config
from portfolio_lab.pipeline import run_analysis, export_analysis

fred_key = os.environ.get("FRED_API_KEY", "")
if USE_FRED and not fred_key:
    try:
        from google.colab import userdata
        fred_key = userdata.get("FRED_API_KEY")
    except Exception:
        raise RuntimeError("Set FRED_API_KEY in Colab Secrets or the environment.") from None

config = Config(start=START, end=END, weights=WEIGHTS,
                benchmark_weights=BENCHMARK_WEIGHTS, initial_capital=INITIAL_CAPITAL,
                rebalance=REBALANCE, cost_bps=COST_BPS, fixed_rf_annual=FIXED_RF_ANNUAL)
analysis = run_analysis(config, mode=DATA_MODE, csv_path=CSV_PATH,
                        cache_dir=PROJECT_ROOT / "data/cache", use_fred=USE_FRED,
                        fred_api_key=fred_key, refresh=REFRESH_CACHE)
print("Data source:", analysis.metadata["data"]["source"])
print("Quality report:", analysis.metadata["data_quality"])
print("Risk-free convention:", analysis.metadata["risk_free"])


In [ ]:
# @title Inspect metrics and reconcile the accounting
import numpy as np
import pandas as pd

try:
    from IPython.display import display, HTML, Image
    NOTEBOOK_DISPLAY = True
except ImportError:
    NOTEBOOK_DISPLAY = False
    def display(value):
        print(value)

display(analysis.metrics.round(4))
portfolio = analysis.results["Portfolio"]
np.testing.assert_allclose(portfolio.contributions.iloc[1:].sum(axis=1),
                           portfolio.returns, atol=2e-12)
np.testing.assert_allclose(analysis.attribution.sum(),
                           portfolio.nav.iloc[-1] - portfolio.nav.iloc[0], atol=1e-7)
display(analysis.attribution.to_frame())
display(analysis.risk.round(4))
print("Daily returns and cumulative dollar P&L reconcile.")


In [ ]:
# @title Export the dashboard, charts, tables and manifest
paths = export_analysis(analysis, PROJECT_ROOT / "reports")
print("Standalone dashboard:", paths["dashboard"])
print("Provenance:", paths["manifest"])
if NOTEBOOK_DISPLAY:
    display(Image(filename=str(paths["executive_summary"])))
else:
    print("Executive summary image:", paths["executive_summary"])


In [ ]:
# @title Preview the interactive dashboard
from html import escape

if NOTEBOOK_DISPLAY:
    report_html = paths["dashboard"].read_text(encoding="utf-8")
    iframe = '<iframe title="Portfolio Lab dashboard" sandbox="allow-scripts" ' + (
        'style="width:100%;height:1000px;border:1px solid #dce3ea;border-radius:12px" srcdoc="'
        + escape(report_html, quote=True) + '"></iframe>')
    display(HTML(iframe))
else:
    print("Open the exported dashboard.html in a browser.")


In [ ]:
# @title Run a transaction-cost sensitivity experiment
from dataclasses import replace
from portfolio_lab.engine import backtest
from portfolio_lab.analytics import summary_metrics

sensitivity_rows = []
for cost in (0, 5, 10, 25):
    result = backtest(analysis.prices, config.weights, config.initial_capital,
                      config.rebalance, cost, name=f"{cost} bps")
    measured = summary_metrics(result, analysis.results["Benchmark"], analysis.risk_free,
                               config.annualization, config.confidence)
    sensitivity_rows.append({"cost_bps": cost, "cagr": measured["cagr"],
                             "ending_nav": measured["ending_nav"], "fees": measured["total_fees"]})
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(PROJECT_ROOT / "reports/cost_sensitivity.csv", index=False)
display(sensitivity)


In [ ]:
# @title Package the source and results for download
import shutil
from zipfile import ZipFile, ZIP_DEFLATED

shutil.copyfile(paths["executive_summary"], PROJECT_ROOT / "examples/executive_summary.png")
bundle_path = PROJECT_ROOT.parent / "Portfolio_Lab_Results.zip"
with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as archive:
    for file in sorted(PROJECT_ROOT.rglob("*")):
        if not file.is_file() or "__pycache__" in file.parts or ".ipynb_checkpoints" in file.parts:
            continue
        if "data/cache" in file.as_posix():
            continue
        archive.write(file, file.relative_to(PROJECT_ROOT))
print("Results and reusable source:", bundle_path)
print("Dashboard:", paths["dashboard"])
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print("Local run: open the bundle path printed above.")
    else:
        files.download(str(bundle_path))
